# 📊 Notebook 00 — Guía metodológica de RADAR Cibest

### *Ranking de Atractivo y Diagnóstico Analítico Regional*

---

**Proyecto:** RADAR Cibest — Técnicas de Selección Internacional de Mercados  
**Autor:** Jhon Adarve — Dirección de Estrategia  
**Audiencia:** este notebook está escrito para que **cualquier lector pueda entenderlo**, sea o no técnico.

---

## ¿Para qué sirve este notebook?

Cuando alguien pregunta *"¿cómo sabe RADAR Cibest que Panamá es más atractivo que Bolivia para nuestra línea de pagos y flujos?"*, este notebook responde la pregunta de principio a fin. Aquí se explica, en orden y con ejemplos, cada técnica que el sistema aplica: cómo se eligieron las dimensiones, cómo se ponderan, cómo se rankea cada país, cómo se incorpora la afinidad con Colombia, cómo se generan las señales por línea de negocio y cómo se valida la robustez del resultado.

La idea es que cualquier persona del Comité Ejecutivo o de la Junta Directiva pueda abrirlo, leerlo de corrido como un libro, y salir entendiendo qué hay detrás de cada cifra del reporte final. Para los analistas del equipo, sirve también como referencia técnica: todas las fórmulas están escritas formalmente y todos los pasos están reproducidos en código ejecutable.

## Cómo leerlo

El notebook se divide en diez capítulos. Los primeros tres dan el contexto del problema y el marco de trabajo. Los capítulos cuatro al ocho explican cada técnica analítica en el orden exacto en que el sistema la aplica. Los capítulos nueve y diez muestran cómo se valida la solidez del modelo y cierran con una recapitulación visual del flujo completo.

Cada capítulo metodológico tiene la misma estructura interna: primero una *intuición* en lenguaje cotidiano, luego la *fórmula* formal, después un *ejemplo numérico* con cinco países que se pueden seguir con calculadora si se quiere, y al final una *visualización interactiva* que permite jugar con los valores.

---

In [1]:
# Configuración inicial del notebook

#Antes de empezar el recorrido cargamos las librerías necesarias y fijamos la paleta corporativa de Cibest para que todos los gráficos compartan el mismo lenguaje visual. Si esta celda corre sin errores significa que el entorno está listo.

# ---------------------------------------------------------------------------
# Importacion de librerias
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from matplotlib.colors import LinearSegmentedColormap
from plotly.subplots import make_subplots
from IPython.display import HTML, display, Markdown
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Paleta corporativa Cibest
# ---------------------------------------------------------------------------
CIBEST = {
    'gray':        '#2C2A28',   # gris profundo institucional
    'gray_light':  '#CCCAC7',   # gris claro para acentos
    'yellow':      '#FDD923',   # acento amarillo Cibest
    'gold':        '#D6B302',   # dorado caracteristico Cibest
    'gold_light':  '#FFF7D3',   # dorado muy claro para fondos y degradados
    'gold_dark':   '#8F7701',   # dorado profundo
    'gray_bg':     '#F5F5F5',   # fondo gris claro
    'gray_border': '#D0D0D0',   # bordes neutros
    'white':       '#FFFFFF',
    'green':       '#0BA682',   # senal alta oportunidad
    'amber':       '#FF7E41',   # senal baja oportunidad
    'red':         '#C62828',   # senal de riesgo

}

SIGNAL_COLORS = {
    'ALTA_OPORTUNIDAD':     CIBEST['green'],
    'OPORTUNIDAD_MODERADA': CIBEST['gold'],
    'BAJA_OPORTUNIDAD':     CIBEST['amber'],
    'RIESGO':               CIBEST['red'],
}

# Template global de Plotly
PLOTLY_TEMPLATE = dict(
    layout=dict(
        font=dict(family='Arial, sans-serif', size=13, color=CIBEST['gray']),
        title=dict(font=dict(size=17, color=CIBEST['gray'])),
        plot_bgcolor=CIBEST['white'],
        paper_bgcolor=CIBEST['white'],
        xaxis=dict(gridcolor=CIBEST['gray_border'], linecolor=CIBEST['gray']),
        yaxis=dict(gridcolor=CIBEST['gray_border'], linecolor=CIBEST['gray']),
        colorway=[CIBEST['gray'], CIBEST['gold'], CIBEST['gray_light'],
                  CIBEST['gold_dark'], CIBEST['green'], CIBEST['amber']],
    )
)

# Degradado personalizado de gris claro a amarillo para heatmaps
cmap_custom = LinearSegmentedColormap.from_list("GrayToYellow", [CIBEST["gray_light"], CIBEST["yellow"]])

# Estilo de tablas pandas
def style_table(df, gradient_cols=None, gradient_cmap='YlGnBu', format_dict=None):
    """Aplica el estilo corporativo Cibest a un DataFrame."""
    styler = df.style.set_table_styles([
        {'selector': 'th',
         'props': [('background-color', CIBEST['gray']),
                   ('color', CIBEST['yellow']),
                   ('font-weight', 'bold'),
                   ('text-align', 'center'),
                   ('padding', '8px'),
                   ('font-family', 'Arial, sans-serif')]},
        {'selector': 'td',
         'props': [('padding', '6px 10px'),
                   ('font-family', 'Arial, sans-serif'),
                   ('border-bottom', f"1px solid {CIBEST['gray_border']}")]},
        {'selector': 'tbody tr:hover',
         'props': [('background-color', CIBEST['gray_bg'])]},
    ])
    if gradient_cols:
        styler = styler.background_gradient(subset=gradient_cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler

#print('Entorno listo — paleta corporativa Cibest activada')


---

# Capítulo 1 — El problema de negocio que RADAR Cibest resuelve

## La pregunta que origina todo

Grupo Cibest tiene operación en Colombia y mira hacia afuera. La región de América Latina, el Caribe, Norteamérica y España ofrece varios mercados potenciales, pero no todos son igualmente atractivos para entrar, ni son atractivos por las mismas razones, ni lo son para todas las líneas de negocio del holding. **Panamá puede ser excelente para Pagos y Flujos gracias a su rol de hub financiero y su diáspora, pero ser solo intermedio para Banca Digital porque el mercado es pequeño en términos de población.** Esa es la complejidad que el sistema tiene que capturar.

La pregunta que la Dirección de Estrategia necesita responder es engañosamente simple: *dados estos 30 mercados candidatos, ¿en qué orden debemos priorizarlos, y por qué razones, y qué tan robusta es esa priorización ante cambios en nuestras prioridades estratégicas?* Responderla bien requiere muchas cosas a la vez: integrar datos heterogéneos de fuentes distintas (Banco Mundial, FMI, indicadores de gobernanza, etc.), incorporar el juicio experto de la alta dirección sobre qué dimensiones importan más, generar señales diferenciadas para cada una de las cinco líneas de negocio, y producir un resultado que se pueda explicar al Comité y a la Junta sin pedirles que estudien optimización multicriterio.

## Las cinco líneas que el sistema atiende

Grupo Cibest agrupa su negocio en cinco líneas, y cada una busca cosas diferentes en un mercado destino. La **Intermediación Bancaria (IB)** es el negocio tradicional de captar depósitos y colocar crédito, y le importa sobre todo la profundidad financiera y la solidez institucional. **Pagos y Flujos (PF)** procesa tarjetas, recaudos y remesas, y necesita penetración digital alta y presencia de diáspora colombiana. **Activos Digitales (AD)** se mueve en cripto, stablecoins y tokens regulados, así que la claridad regulatoria es casi todo. **Bancos Digitales (BD)** opera sin sucursales y prospera donde hay grandes mercados jóvenes con alta digitalización. **Corporate & Investment Banking (CIB)** estructura operaciones para grandes corporativos y busca mercados grandes, estables y con buen rating soberano.

La consecuencia metodológica de esto es importante: **no basta con un solo ranking global**. RADAR Cibest produce un ranking global y además cinco rankings adicionales, uno por línea, cada uno con pesos diferenciados sobre las mismas variables.

## Las cinco dimensiones que se evalúan

Para evaluar cualquier mercado el sistema mira cinco dimensiones distintas que juntas capturan tanto el "tamaño del premio" como la "facilidad de capturarlo" y el "riesgo de intentarlo":

In [2]:
# Tabla de las cinco dimensiones
dimensiones = pd.DataFrame([
    {'Dimensión': 'Macroeconómica',     'Qué responde': 'Qué tan grande, dinámico y estable es el mercado',
     'Ejemplos típicos': 'PIB, crecimiento, inflación, desempleo, integracion internacional (apertura comercial e inversión extrajera directa)'},
    {'Dimensión': 'Financiera',         'Qué responde': 'Qué tan profundo, accesible y eficiente es el sistema financiero',
     'Ejemplos típicos': 'Crédito/PIB, depósitos, cuentas bancarias, ratio de préstamos mororsos'},
    {'Dimensión': 'Institucional',      'Qué responde': 'Qué tan confiable es operar ahí en términos legales y regulatorios',
     'Ejemplos típicos': 'WGI, control de corrupción, calidad regulatoria, riesgo soberano'},
    {'Dimensión': 'Digital-Tecnológica','Qué responde': 'Qué tan preparado está el mercado para servicios digitales',
     'Ejemplos típicos': 'Usuarios de internet, móviles, pagos digitales, adopción cripto'},
    {'Dimensión': 'Proximidad',         'Qué responde': 'Qué tan cerca está de Colombia en sentido amplio',
     'Ejemplos típicos': 'Distancia, idioma, distancia cultural, diáspora, comercio bilateral'},
])
style_table(dimensiones)


,Dimensión,Qué responde,Ejemplos típicos
0,Macroeconómica,"Qué tan grande, dinámico y estable es el mercado","PIB, crecimiento, inflación, desempleo, integracion internacional (apertura comercial e inversión extrajera directa)"
1,Financiera,"Qué tan profundo, accesible y eficiente es el sistema financiero","Crédito/PIB, depósitos, cuentas bancarias, ratio de préstamos mororsos"
2,Institucional,Qué tan confiable es operar ahí en términos legales y regulatorios,"WGI, control de corrupción, calidad regulatoria, riesgo soberano"
3,Digital-Tecnológica,Qué tan preparado está el mercado para servicios digitales,"Usuarios de internet, móviles, pagos digitales, adopción cripto"
4,Proximidad,Qué tan cerca está de Colombia en sentido amplio,"Distancia, idioma, distancia cultural, diáspora, comercio bilateral"


La intuición que une las cinco dimensiones es la siguiente: las tres primeras describen *propiedades intrínsecas del mercado destino* (qué tan grande, qué tan desarrollado financieramente, qué tan confiable). La cuarta refleja la *preparación del mercado para los servicios modernos* que ofrece Cibest. La quinta es distinta y muy importante: no describe al mercado destino en sí, sino la *relación bilateral entre Colombia y ese mercado*. Un mercado igualmente atractivo "en abstracto" es más atractivo para Cibest si está cerca, si comparte idioma, si tiene un perfil cultural parecido y si recibe muchos colombianos al año. Esa dimensión bilateral es la que se modela con un *modelo gravitacional* en el capítulo seis.

Cada una de estas cinco dimensiones se descompone en variables observables. Las cinco juntas suman **40 variables** que el sistema extrae periódicamente de cinco fuentes distintas (Banco Mundial, indicadores de gobernanza WGI, prima de riesgo país Damodaran, índice de libertad económica Heritage, y conjuntos de datos complementarios estáticos como CEPII, Hofstede y registros de salidas de colombianos). El detalle completo de las variables se presenta en el capítulo tres, pero por ahora vale la pena tener en mente que la distribución del trabajo entre dimensiones **no es simétrica**: Macroecnomica,Proximidad y Financiera tienen más de nueve variables cada una, mientras que Digital-Tecnológica tiene solo siete debido a las limitadas fuentes y completitud (por la naturaleza descentralizada de las criptomonedas y la novedad de las finanzas abiertas y finctech, muchas de estas métricas no miden transacciones directas, sino que se utilizan "proxys" (indicadores indirectos) sobre adopción, regulación y comportamiento). Esa asimetría es una decisión metodológica deliberada que se explica en su momento, y refleja entre otras cosas que las seis dimensiones culturales de Hofstede se incluyen como variables individuales antes de agregarse en un índice de distancia cultural.

---

# Capítulo 2 — El marco metodológico: ASUM-DM

## ¿Por qué un marco metodológico y no improvisación?

Un proyecto analítico que va a llegar a la Junta Directiva y que va a guiar decisiones de inversión internacional no puede entregarse con una sola corrida exitosa de código. Tiene que ser *reproducible, auditable, robusto y revisable*. Para conseguir eso RADAR Cibest se apoya en una metodología estructurada llamada **ASUM-DM** (*Analytics Solutions Unified Method for Data Mining*), que es la evolución moderna de CRISP-DM adaptada por IBM para proyectos analíticos de gran escala.

ASUM-DM organiza el trabajo en cinco fases secuenciales pero iterativas. *Secuenciales* porque cada fase produce un entregable que alimenta la siguiente. *Iterativas* porque si una fase posterior revela un problema (por ejemplo, descubrir en la fase de modelado que faltan datos críticos), se vuelve atrás a la fase apropiada y se itera. El sistema RADAR está implementado siguiendo este flujo de cinco fases con checkpoints formales al cierre de cada una.

## Las cinco fases de ASUM-DM aplicadas a RADAR

In [3]:
fases = pd.DataFrame([
    {'Fase': '1. Entendimiento del negocio',
     'Pregunta central': '¿Qué problema de negocio resolvemos y cómo se medirá éxito?',
     'Entregable en RADAR': 'Project Charter aprobado, matriz de riesgos, alcance de 30 países'},
    {'Fase': '2. Entendimiento de los datos',
     'Pregunta central': '¿Qué datos tenemos, cuáles necesitamos y qué tan buenos son?',
     'Entregable en RADAR': 'Revisión sistemática de literatura, diccionario de variables, mapa de fuentes'},
    {'Fase': '3. Preparación de los datos',
     'Pregunta central': '¿Cómo transformamos los datos crudos en una matriz lista para modelar?',
     'Entregable en RADAR': 'Pipeline de extracción, limpieza, imputación, normalización con dirección'},
    {'Fase': '4. Modelado',
     'Pregunta central': '¿Qué técnicas aplicamos y cómo las integramos?',
     'Entregable en RADAR': 'BWM (pesos) + TOPSIS (ranking) + Gravitacional (proximidad) + Score compuesto'},
    {'Fase': '5. Evaluación',
     'Pregunta central': '¿Qué tan robusto y útil es el resultado?',
     'Entregable en RADAR': 'Análisis de sensibilidad, validación cruzada TOPSIS-VIKOR, reporte ejecutivo'},
])
style_table(fases)


,Fase,Pregunta central,Entregable en RADAR
0,1. Entendimiento del negocio,¿Qué problema de negocio resolvemos y cómo se medirá éxito?,"Project Charter aprobado, matriz de riesgos, alcance de 30 países"
1,2. Entendimiento de los datos,"¿Qué datos tenemos, cuáles necesitamos y qué tan buenos son?","Revisión sistemática de literatura, diccionario de variables, mapa de fuentes"
2,3. Preparación de los datos,¿Cómo transformamos los datos crudos en una matriz lista para modelar?,"Pipeline de extracción, limpieza, imputación, normalización con dirección"
3,4. Modelado,¿Qué técnicas aplicamos y cómo las integramos?,BWM (pesos) + TOPSIS (ranking) + Gravitacional (proximidad) + Score compuesto
4,5. Evaluación,¿Qué tan robusto y útil es el resultado?,"Análisis de sensibilidad, validación cruzada TOPSIS-VIKOR, reporte ejecutivo"


Vale la pena hacer una observación pedagógica importante aquí: **las técnicas analíticas concretas (BWM, TOPSIS, modelo gravitacional) aparecen únicamente en la fase 4**. Las tres primeras fases no tienen "matemática" en sentido estricto; son trabajo de definición, comprensión y preparación. Pero un proyecto que se salta esas tres fases para llegar rápido a la técnica suele producir resultados técnicamente funcionales pero conceptualmente equivocados. La calidad final de RADAR Cibest depende tanto del rigor de la fase 4 como de la solidez de las fases 1, 2 y 3 que la preceden.

## Cómo encaja todo en el flujo del sistema

In [4]:
import plotly.graph_objects as go

# ============================================================
# Diagrama del flujo del sistema RADAR
# Proyecto analítico de selección internacional de mercados
# ============================================================

fig = go.Figure()


# ------------------------------------------------------------
# Definición de nodos
# x, y controlan la posición.
# w y h controlan tamaño visual de cada caja.
# ------------------------------------------------------------
nodos = [
    {
        "id": "fuentes",
        "texto": "Fuentes de datos<br><span style='font-size:11px'>WB, FMI, WGI, ITC, etc.</span>",
        "x": 0, "y": 4,
        "color": CIBEST["gray_bg"],
        "font": CIBEST["gray"],
        "grupo": "Datos"
    },
    {
        "id": "extraccion",
        "texto": "Extracción<br><span style='font-size:11px'>APIs y CSV estáticos</span>",
        "x": 1.5, "y": 4,
        "color": CIBEST["gray_light"],
        "font": CIBEST["gray"],
        "grupo": "Datos"
    },
    {
        "id": "preparacion",
        "texto": "Preparación<br><span style='font-size:11px'>Limpieza, imputación<br>y normalización</span>",
        "x": 3.0, "y": 4,
        "color": CIBEST["gray_light"],
        "font": CIBEST["gray"],
        "grupo": "Datos"
    },
    {
        "id": "matriz",
        "texto": "Matriz de decisión<br><span style='font-size:11px'>País × variable</span>",
        "x": 4.5, "y": 4,
        "color": CIBEST["gold_light"],
        "font": CIBEST["gray"],
        "grupo": "Modelo"
    },

    {
        "id": "bwm",
        "texto": "BWM<br><span style='font-size:11px'>Pesos del panel</span>",
        "x": 6.2, "y": 5.4,
        "color": CIBEST["gray"],
        "font": "white",
        "grupo": "Modelo"
    },
    {
        "id": "topsis",
        "texto": "TOPSIS<br><span style='font-size:11px'>Ranking competitivo</span>",
        "x": 6.2, "y": 4,
        "color": CIBEST["gray"],
        "font": "white",
        "grupo": "Modelo"
    },
    {
        "id": "gravitacional",
        "texto": "Modelo gravitacional<br><span style='font-size:11px'>IPC / potencial comercial</span>",
        "x": 6.2, "y": 2.6,
        "color": CIBEST["gray"],
        "font": "white",
        "grupo": "Modelo"
    },

    {
        "id": "score",
        "texto": "Score RADAR compuesto<br><span style='font-size:11px'>α·CC + β·IPC + γ·Tendencia</span>",
        "x": 8.2, "y": 4,
        "color": CIBEST["gold"],
        "font": CIBEST["gray"],
        "grupo": "Decisión"
    },
    {
        "id": "senales",
        "texto": "Motor de señales<br><span style='font-size:11px'>4 niveles × 5 líneas</span>",
        "x": 10.0, "y": 4,
        "color": CIBEST["gold_dark"],
        "font": "white",
        "grupo": "Decisión"
    },
    {
        "id": "reporte",
        "texto": "Reporte estratégico<br><span style='font-size:11px'>Dashboard + narrativa ejecutiva</span>",
        "x": 11.8, "y": 4,
        "color": CIBEST["green"],
        "font": "white",
        "grupo": "Salida"
    },
]

# Diccionario de posiciones para conectores
pos = {n["id"]: (n["x"], n["y"]) for n in nodos}

# ------------------------------------------------------------
# Función para crear cajas como anotaciones
# Esto evita problemas de texto sobre marcadores circulares
# ------------------------------------------------------------
def agregar_caja(fig, x, y, texto, color, font_color, ancho=1.25, alto=0.62):
    fig.add_annotation(
        x=x,
        y=y,
        text=texto,
        showarrow=False,
        align="center",
        font=dict(
            size=13,
            color=font_color,
            family="Arial"
        ),
        bgcolor=color,
        bordercolor=CIBEST["gray"],
        borderwidth=1.2,
        borderpad=8,
        opacity=1
    )

# ------------------------------------------------------------
# Función para agregar conectores
# Se usan desplazamientos para que las flechas no entren
# directamente al centro de las cajas
# ------------------------------------------------------------
def agregar_flecha(fig, origen, destino, color=CIBEST["gray"]):
    x0, y0 = pos[origen]
    x1, y1 = pos[destino]

    # Ajuste horizontal para que la flecha salga/entre por bordes
    dx = 0.52 if x1 >= x0 else -0.52
    

    fig.add_annotation(
        x=x1 - dx,
        y=y1,
        ax=x0 + dx,
        ay=y0,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor=color,
        opacity=0.85
    )

# ------------------------------------------------------------
# Bandas de fondo por fase
# ------------------------------------------------------------
fases = [
    {
        "nombre": "1. Datos",
        "x0": -0.7, "x1": 3.75,
        "color": "#F8F9FA"
    },
    {
        "nombre": "2. Modelamiento analítico",
        "x0": 3.75, "x1": 7.35,
        "color": "#FFF8E8"
    },
    {
        "nombre": "3. Priorización y señales",
        "x0": 7.35, "x1": 10.9,
        "color": "#F7F2E8"
    },
    {
        "nombre": "4. Salida ejecutiva",
        "x0": 10.9, "x1": 12.6,
        "color": "#EAF6EF"
    }
]

for fase in fases:
    fig.add_shape(
        type="rect",
        x0=fase["x0"], x1=fase["x1"],
        y0=1.7, y1=6.15,
        line=dict(width=0),
        fillcolor=fase["color"],
        layer="below"
    )

    fig.add_annotation(
        x=(fase["x0"] + fase["x1"]) / 2,
        y=6.0,
        text=f"<b>{fase['nombre']}</b>",
        showarrow=False,
        font=dict(size=12, color=CIBEST["gray"]),
        align="center"
    )

# ------------------------------------------------------------
# Agregar nodos
# ------------------------------------------------------------
for n in nodos:
    agregar_caja(
        fig=fig,
        x=n["x"],
        y=n["y"],
        texto=n["texto"],
        color=n["color"],
        font_color=n["font"]
    )

# ------------------------------------------------------------
# Conectores principales
# ------------------------------------------------------------
conectores = [
    ("fuentes", "extraccion"),
    ("extraccion", "preparacion"),
    ("preparacion", "matriz"),

    ("matriz", "bwm"),
    ("matriz", "topsis"),
    ("matriz", "gravitacional"),

    ("bwm", "score"),
    ("topsis", "score"),
    ("gravitacional", "score"),

    ("score", "senales"),
    ("senales", "reporte"),
]

for origen, destino in conectores:
    agregar_flecha(fig, origen, destino)

# ------------------------------------------------------------
# Nota metodológica inferior
# ------------------------------------------------------------
fig.add_annotation(
    x=5.9,
    y=1.35,
    text=(
        "El sistema RADAR integra fuentes internacionales, técnicas multicriterio "
        "y señales estratégicas para priorizar mercados con mayor atractivo y viabilidad."
    ),
    showarrow=False,
    font=dict(size=11, color=CIBEST["gray"]),
    align="center"
)

# ------------------------------------------------------------
# Layout general
# ------------------------------------------------------------
fig.update_layout(
    title=dict(
        text=(
            "<b>Flujo del sistema RADAR Cibest</b>"
            "<br><sup>Proyecto analítico de selección internacional de mercados</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=22, color=CIBEST["gray"])
    ),
    xaxis=dict(
        visible=False,
        range=[-0.8, 12.7],
        fixedrange=True
    ),
    yaxis=dict(
        visible=False,
        range=[1.1, 6.45],
        fixedrange=True
    ),
    height=560,
    width=1250,
    plot_bgcolor=CIBEST["white"],
    paper_bgcolor=CIBEST["white"],
    font=dict(
        family="Arial",
        color=CIBEST["gray"]
    ),
    margin=dict(t=95, b=40, l=30, r=30)
)

fig.show()

La imagen anterior es el mapa mental del notebook completo. En los capítulos siguientes vamos a abrir cada uno de esos bloques uno por uno. Si en algún momento se pierde el hilo, esta es la figura a la que conviene regresar.

---

# Capítulo 3 — Las dimensiones y sus variables

## De dimensiones a variables medibles

En el capítulo uno presentamos las cinco dimensiones del análisis (Macro, Financiera, Institucional, Digital-Tecnológica y Proximidad). Pero esas dimensiones son *conceptos*, no son cosas que se puedan medir directamente. Para que el sistema pueda trabajar con ellas hay que descomponer cada una en *variables observables*, cada una con su fuente de datos, su unidad, su rango típico y su **dirección**.

La dirección de una variable es un concepto sutil pero crucial. Decir que una variable tiene *dirección positiva* significa que **valores más altos son mejores para el atractivo del mercado**. El PIB per cápita tiene dirección positiva: más alto es mejor. Decir que una variable tiene *dirección negativa* significa que valores más altos son peores. La inflación tiene dirección negativa: más alta es peor. La distancia geográfica a Bogotá también tiene dirección negativa: más lejos es peor (en términos de proximidad con Colombia). Sin clasificar bien la dirección de cada variable el sistema invertiría el sentido del ranking en algunas dimensiones, y producir un resultado erróneo.

Hay un tercer caso, sutilmente distinto: la dirección **neutral**. Una variable es neutral cuando ni "mucho" ni "poco" son mejores en términos absolutos; lo que importa es **la similitud con Colombia**. Las seis dimensiones culturales de Hofstede son el ejemplo perfecto: no es mejor tener un Power Distance alto o bajo en sentido absoluto; es mejor que el Power Distance del país destino sea *parecido al de Colombia*, porque la similitud cultural facilita la operación y reduce los costos de adaptación. El tratamiento de las variables neutrales requiere un paso de transformación adicional que veremos al final del capítulo: se calcula la **distancia cultural Kogut-Singh** entre cada destino y Colombia agregando las seis dimensiones Hofstede en un solo índice, y esa distancia agregada sí tiene dirección negativa (menor distancia es mejor).

## El diccionario completo de variables

A continuación se presenta el diccionario completo del proyecto. La tabla está organizada por dimensión y dentro de cada dimensión por orden de prioridad. Para cada variable se documenta el nombre técnico (usado en el código), la descripción en lenguaje natural, la fuente, el código del indicador en la fuente, la dirección y la frecuencia de actualización.

In [5]:
# Diccionario completo de las 45 variables del proyecto RADAR Cibest
# Fuente: Excel "RADAR Diccionario Variables Actualizado" - version definitiva 2026


diccionario = pd.DataFrame([

    # ===== MACROECONOMICA - 12 variables =====
    {'Dim': 'Macro', '#':  1, 'Variable': 'gdp_nominal', 'Descripcion': 'PIB nominal (USD corrientes)',
     'Fuente': 'World Bank', 'Codigo': 'NY.GDP.MKTP.CD', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  2, 'Variable': 'gdp_per_capita_ppp', 'Descripcion': 'PIB per capita PPP (USD intl.)',
     'Fuente': 'World Bank', 'Codigo': 'NY.GDP.PCAP.PP.CD', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  3, 'Variable': 'gdp_growth', 'Descripcion': 'Crecimiento real anual del PIB',
     'Fuente': 'World Bank', 'Codigo': 'NY.GDP.MKTP.KD.ZG', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  4, 'Variable': 'inflation_rate', 'Descripcion': 'Inflacion anual IPC',
     'Fuente': 'World Bank', 'Codigo': 'FP.CPI.TOTL.ZG', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  5, 'Variable': 'population_total', 'Descripcion': 'Poblacion total',
     'Fuente': 'World Bank', 'Codigo': 'SP.POP.TOTL', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  6, 'Variable': 'urban_population_pct', 'Descripcion': 'Poblacion urbana (% del total)',
     'Fuente': 'World Bank', 'Codigo': 'SP.URB.TOTL.IN.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  7, 'Variable': 'unemployment_rate', 'Descripcion': 'Desempleo total (% fuerza laboral)',
     'Fuente': 'World Bank', 'Codigo': 'SL.UEM.TOTL.ZS', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  8, 'Variable': 'current_account_gdp', 'Descripcion': 'Cuenta corriente / PIB (%)',
     'Fuente': 'World Bank', 'Codigo': 'BN.CAB.XOKA.GD.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#':  9, 'Variable': 'public_debt_gdp', 'Descripcion': 'Deuda publica / PIB (%)',
     'Fuente': 'World Bank', 'Codigo': 'GC.DOD.TOTL.GD.ZS', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#': 10, 'Variable': 'trade_openness', 'Descripcion': 'Comercio total (% PIB)',
     'Fuente': 'World Bank', 'Codigo': 'NE.TRD.GNFS.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#': 11, 'Variable': 'fdi_net_inflows_gdp', 'Descripcion': 'IED neta / PIB (%)',
     'Fuente': 'World Bank', 'Codigo': 'BX.KLT.DINV.WD.GD.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Macro', '#': 12, 'Variable': 'weighted_mean_applied_tariff_all_products', 'Descripcion': 'Tarifa aplicada promedio ponderada (%)',
     'Fuente': 'World Bank', 'Codigo': 'TM.TAX.MRCH.WM.AR.ZS', 'Direccion': 'negative', 'Frecuencia': 'Anual'},


    # ===== FINANCIERA - 9 variables =====
    {'Dim': 'Financ.', '#': 13, 'Variable': 'domestic_credit_private_gdp',
     'Descripcion': 'Credito al sector privado (% PIB)', 'Fuente': 'World Bank',
     'Codigo': 'FD.AST.PRVT.GD.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 14, 'Variable': 'account_ownership',
     'Descripcion': 'Adultos con cuenta', 'Fuente': 'World Bank',
     'Codigo': 'FX.OWN.TOTL.ZS', 'Direccion': 'positive', 'Frecuencia': 'Trienal'},

    {'Dim': 'Financ.', '#': 15, 'Variable': 'interest_rate_spread',
     'Descripcion': 'Spread tasa activa - pasiva', 'Fuente': 'World Bank',
     'Codigo': 'FR.INR.LNDP', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 16, 'Variable': 'bank_npl_ratio',
     'Descripcion': 'Cartera vencida (%)', 'Fuente': 'World Bank',
     'Codigo': 'FB.AST.NPER.ZS', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 17, 'Variable': 'stock_market_cap_gdp',
     'Descripcion': 'Capitalizacion bursatil (% PIB)', 'Fuente': 'World Bank',
     'Codigo': 'CM.MKT.LCAP.GD.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 18, 'Variable': 'personal_remittances_gdp',
     'Descripcion': 'Remesas (% PIB)', 'Fuente': 'World Bank',
     'Codigo': 'BX.TRF.PWKR.DT.GD.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 19, 'Variable': 'bank_concentration_5',
     'Descripcion': 'Concentracion top-5 bancos', 'Fuente': 'World Bank',
     'Codigo': 'GFDD.OI.06', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 20, 'Variable': 'financial_system_deposits_gc',
     'Descripcion': 'Depositos / PIB', 'Fuente': 'World Bank',
     'Codigo': 'GFDD.DI.08', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Financ.', '#': 21, 'Variable': 'bank_capital_rwa',
     'Descripcion': 'Capital regulatorio / RWA', 'Fuente': 'World Bank',
     'Codigo': 'GFDD.SI.05', 'Direccion': 'positive', 'Frecuencia': 'Anual'},


    # ===== INSTITUCIONAL - 8 variables =====
    {'Dim': 'Instit.', '#': 22, 'Variable': 'regulatory_quality', 'Descripcion': 'WGI: calidad regulatoria',
     'Fuente': 'World Bank', 'Codigo': 'GOV_WGI_RQ.EST', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 23, 'Variable': 'government_effectiveness', 'Descripcion': 'WGI: efectividad gubernamental',
     'Fuente': 'World Bank', 'Codigo': 'GOV_WGI_GE.EST', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 24, 'Variable': 'rule_of_law', 'Descripcion': 'WGI: estado de derecho',
     'Fuente': 'World Bank', 'Codigo': 'GOV_WGI_RL.EST', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 25, 'Variable': 'political_stability', 'Descripcion': 'WGI: estabilidad politica',
     'Fuente': 'World Bank', 'Codigo': 'GOV_WGI_PV.EST', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 26, 'Variable': 'voice_accountability', 'Descripcion': 'WGI: voz y rendicion de cuentas',
     'Fuente': 'World Bank', 'Codigo': 'GOV_WGI_VA.EST', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 27, 'Variable': 'control_of_corruption', 'Descripcion': 'WGI: control de corrupcion',
     'Fuente': 'World Bank', 'Codigo': 'GOV_WGI_CC.EST', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 28, 'Variable': 'country_risk_premium', 'Descripcion': 'Prima de riesgo pais',
     'Fuente': 'Damodaran', 'Codigo': 'COUNTRY_RISK_PREMIUM', 'Direccion': 'negative', 'Frecuencia': 'Anual'},

    {'Dim': 'Instit.', '#': 29, 'Variable': 'heritage_efi', 'Descripcion': 'Economic Freedom Index',
     'Fuente': 'Heritage', 'Codigo': 'HERITAGE_EFI', 'Direccion': 'positive', 'Frecuencia': 'Anual'},


    # ===== DIGITAL - 7 variables =====
    {'Dim': 'Digital', '#': 30, 'Variable': 'internet_users_pct', 'Descripcion': 'Usuarios de internet',
     'Fuente': 'World Bank', 'Codigo': 'IT.NET.USER.ZS', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Digital', '#': 31, 'Variable': 'mobile_subscriptions', 'Descripcion': 'Suscripciones moviles',
     'Fuente': 'World Bank', 'Codigo': 'IT.CEL.SETS.P2', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Digital', '#': 32, 'Variable': 'digital_payment_adults_pct', 'Descripcion': 'Pagos digitales',
     'Fuente': 'World Bank', 'Codigo': 'g20.any', 'Direccion': 'positive', 'Frecuencia': 'Trienal'},

    {'Dim': 'Digital', '#': 33, 'Variable': 'secure_internet_servers', 'Descripcion': 'Servidores seguros',
     'Fuente': 'World Bank', 'Codigo': 'IT.NET.SECR.P6', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Digital', '#': 34, 'Variable': 'bank_branches', 'Descripcion': 'Sucursales bancarias',
     'Fuente': 'World Bank', 'Codigo': 'FB.CBK.BRCH.P5', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Digital', '#': 35, 'Variable': 'atm_density', 'Descripcion': 'Cajeros automaticos',
     'Fuente': 'World Bank', 'Codigo': 'FB.ATM.TOTL.P5', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    {'Dim': 'Digital', '#': 36, 'Variable': 'ict_exports_pct', 'Descripcion': 'Exportaciones TIC',
     'Fuente': 'World Bank', 'Codigo': 'TX.VAL.ICTG.ZS.UN', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

    # ===== PROXIMIDAD - 9 variables =====
    {'Dim': 'Proxim.', '#': 37, 'Variable': 'geographic_distance_km', 'Descripcion': 'Distancia a Bogota',
     'Fuente': 'CEPII', 'Codigo': 'CEPII_DIST_BOG', 'Direccion': 'negative', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 38, 'Variable': 'common_language_spanish', 'Descripcion': 'Idioma comun',
     'Fuente': 'CEPII', 'Codigo': 'CEPII_LANG_ES', 'Direccion': 'positive', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 39, 'Variable': 'hofstede_pdi', 'Descripcion': 'Power distance',
     'Fuente': 'Hofstede', 'Codigo': 'pdi', 'Direccion': 'neutral', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 40, 'Variable': 'hofstede_idv', 'Descripcion': 'Individualism',
     'Fuente': 'Hofstede', 'Codigo': 'idv', 'Direccion': 'neutral', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 41, 'Variable': 'hofstede_mas', 'Descripcion': 'Masculinity',
     'Fuente': 'Hofstede', 'Codigo': 'mas', 'Direccion': 'neutral', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 42, 'Variable': 'hofstede_uai', 'Descripcion': 'Uncertainty avoidance',
     'Fuente': 'Hofstede', 'Codigo': 'uai', 'Direccion': 'neutral', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 43, 'Variable': 'hofstede_lto', 'Descripcion': 'Long term orientation',
     'Fuente': 'Hofstede', 'Codigo': 'lto', 'Direccion': 'neutral', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 44, 'Variable': 'hofstede_ivr', 'Descripcion': 'Indulgence',
     'Fuente': 'Hofstede', 'Codigo': 'ivr', 'Direccion': 'neutral', 'Frecuencia': 'Estatica'},

    {'Dim': 'Proxim.', '#': 45, 'Variable': 'colombian_diaspora_stock', 'Descripcion': 'Flujo migratorio colombiano',
     'Fuente': 'Migracion', 'Codigo': 'SALIDAS_COL', 'Direccion': 'positive', 'Frecuencia': 'Anual'},

])

# Funcion para colorear la columna 'Direccion' con codigos visuales
def colorear_direccion(val):
    colors = {
        'positive': f'background-color: {CIBEST["green"]}; color: white; font-weight: 600',
        'negative': f'background-color: {CIBEST["red"]}; color: white; font-weight: 600',
        'neutral':  f'background-color: {CIBEST["gold"]}; color: {CIBEST["gray"]}; font-weight: 600',
    }
    return colors.get(val, '')

# Tabla estilizada
styled = diccionario.style.set_table_styles([
    {'selector': 'th',
     'props': [('background-color', CIBEST['gray']),
               ('color', CIBEST['yellow']),
               ('font-weight', 'bold'),
               ('text-align', 'center'),
               ('padding', '8px'),
               ('font-family', 'Arial, sans-serif'),
               ('font-size', '12px')]},
    {'selector': 'td',
     'props': [('padding', '5px 8px'),
               ('font-family', 'Arial, sans-serif'),
               ('font-size', '11px'),
               ('border-bottom', f"1px solid {CIBEST['gray_border']}")]},
    {'selector': 'tbody tr:hover',
     'props': [('background-color', CIBEST['gray_bg'])]},
]).map(colorear_direccion, subset=['Direccion']).hide(axis='index')

display(styled)


Dim,#,Variable,Descripcion,Fuente,Codigo,Direccion,Frecuencia
Macro,1,gdp_nominal,PIB nominal (USD corrientes),World Bank,NY.GDP.MKTP.CD,positive,Anual
Macro,2,gdp_per_capita_ppp,PIB per capita PPP (USD intl.),World Bank,NY.GDP.PCAP.PP.CD,positive,Anual
Macro,3,gdp_growth,Crecimiento real anual del PIB,World Bank,NY.GDP.MKTP.KD.ZG,positive,Anual
Macro,4,inflation_rate,Inflacion anual IPC,World Bank,FP.CPI.TOTL.ZG,negative,Anual
Macro,5,population_total,Poblacion total,World Bank,SP.POP.TOTL,positive,Anual
Macro,6,urban_population_pct,Poblacion urbana (% del total),World Bank,SP.URB.TOTL.IN.ZS,positive,Anual
Macro,7,unemployment_rate,Desempleo total (% fuerza laboral),World Bank,SL.UEM.TOTL.ZS,negative,Anual
Macro,8,current_account_gdp,Cuenta corriente / PIB (%),World Bank,BN.CAB.XOKA.GD.ZS,positive,Anual
Macro,9,public_debt_gdp,Deuda publica / PIB (%),World Bank,GC.DOD.TOTL.GD.ZS,negative,Anual
Macro,10,trade_openness,Comercio total (% PIB),World Bank,NE.TRD.GNFS.ZS,positive,Anual


## Visualización del peso de cada dimensión

Una observación que salta a la vista en el diccionario es que **las dimensiones no tienen el mismo número de variables**. La dimensión Digital-Tecnológica tiene 7 variables e instritucional 8, mientras que las dimensiones Financiera y Proximidad tienen nueve cada una. La gráfica siguiente lo hace explícito:

In [6]:
# Conteo de variables por dimension
conteo_dim = diccionario.groupby('Dim').size().reset_index(name='n_variables')
conteo_dim = conteo_dim.sort_values('n_variables', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=conteo_dim['Dim'], x=conteo_dim['n_variables'],
    orientation='h',
    marker=dict(color=conteo_dim['n_variables'],
                colorscale=[[0, CIBEST['gray_light']], [1, CIBEST['yellow']]],
                #line=dict(color=CIBEST['gray'], width=1.5)
    ), 
    text=conteo_dim['n_variables'],
    textposition='outside',
    textfont=dict(size=14, color=CIBEST['gray'], family='Arial Black'),
    hovertemplate='<b>%{y}</b><br>Variables: %{x}<extra></extra>',
))

fig.update_layout(
    title='<b>Variables por dimensión (45 totales)</b><br>'
          '<sub>La asimetría refleja decisiones metodológicas, no un sesgo</sub>',
    xaxis=dict(title='Número de variables', range=[0, 15], gridcolor=CIBEST['gray_border']),
    yaxis_title='Dimensión',
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=380, width=850, showlegend=False, margin=dict(l=100, t=80),
)
fig.show()


¿Por qué Proximidad tiene más variables que Digital-Tecnológica? La respuesta es metodológica y conviene desglosarla. Seis de las nueve variables de Proximidad son las seis dimensiones culturales de Hofstede. La literatura de proximidad cultural (Kogut & Singh 1988, Ghemawat 2001) recomienda mantener las seis dimensiones por separado durante la extracción y construir el índice de distancia cultural agregada en una etapa posterior, en lugar de promediarlas a priori, porque cada dimensión tiene una varianza distinta entre países y la distancia debe ponderarse por esa varianza. En la práctica, esas seis variables se *agregan* en una única variable derivada llamada `kogut_singh_distance` antes de entrar a TOPSIS, por lo que en términos de poder explicativo Proximidad contribuye con cuatro variables efectivas (la distancia cultural agregada, la distancia geográfica, el idioma compartido y las salidas de colombianos), no con nueve.

Algo análogo pasa con Institucional, donde seis de las ocho variables son los seis indicadores WGI del Banco Mundial. Estos seis indicadores están altamente correlacionados entre sí (un país con mala calidad regulatoria suele también tener débil estado de derecho), por lo que el sistema los mantiene separados para diagnóstico pero su contribución efectiva al ranking es menor que lo que la cuenta sugiere. La dimensión Digital-Tecnológica con sus tres variables, por contraste, captura el fenómeno completo con muy poca redundancia: una variable para la *base instalada* (usuarios de internet), una para el *canal de distribución* (móviles), una para la *adopción específica del producto financiero digital* (pagos digitales) y tres más que denotan la infraestructura teconologica disponible (cajeros, sucursales y servidores seguros).

## Las variables culturales Hofstede y la distancia Kogut-Singh

La explicación más detallada del tratamiento de las seis variables Hofstede merece su propio espacio, porque es el único caso del sistema donde la dirección de las variables no es "positiva" ni "negativa" sino "neutral". La fórmula que el sistema aplica es la **distancia Kogut-Singh**, propuesta por Bruce Kogut y Harbir Singh en 1988 y considerada el estándar de la literatura en estudios de internacionalización. Para cada país $j$ y dimensión cultural $k$ (de las seis de Hofstede), se calcula:

$$
d_{k}(\text{COL}, j) = \frac{(I_{k,j} - I_{k,\text{COL}})^2}{\sigma_k^2}
$$

donde $I_{k,j}$ es el valor de Hofstede del país $j$ en la dimensión $k$, $I_{k,\text{COL}}$ es el valor de Colombia en esa misma dimensión, y $\sigma_k^2$ es la varianza de la dimensión $k$ a lo largo de todos los países. La normalización por varianza es lo que evita que una dimensión con escala más amplia (como Power Distance, que va de 11 a 104) domine sobre dimensiones con menos rango.

Luego se agregan las seis distancias parciales en un solo índice:

$$
\text{KS}(\text{COL}, j) = \frac{1}{6} \sum_{k=1}^{6} d_k(\text{COL}, j)
$$

Esa cantidad $\text{KS}$ es siempre no negativa, es cero cuando el país tiene exactamente el mismo perfil cultural que Colombia (caso teórico que aplica solo a Colombia consigo misma) y crece a medida que el perfil cultural se aleja. **Esa distancia agregada sí tiene dirección negativa** (menor distancia es mejor para Cibest), y es ella la que entra a TOPSIS como variable de proximidad cultural, no las seis variables originales.

## Las fuentes de datos: de dónde sale cada cifra

In [7]:
# Conteo de variables por fuente y descripcion de cada una
fuentes = diccionario.groupby('Fuente').size().reset_index(name='n_variables')
fuentes_info = {
    'World Bank':  {'descripcion': 'World Development Indicators - API publica REST', 'metodo': 'wbgapi (DB=2)'},
    'WB GFDD':     {'descripcion': 'Global Financial Development Database',          'metodo': 'wbgapi (DB=32)'},
    'WGI':         {'descripcion': 'Worldwide Governance Indicators',                'metodo': 'wbgapi (DB=3)'},
    'Damodaran':   {'descripcion': 'NYU Stern Country Risk Premium',                  'metodo': 'requests + Excel'},
    'Heritage':    {'descripcion': 'Heritage Foundation Index of Economic Freedom',  'metodo': 'CSV / API complementaria'},
    'CEPII':       {'descripcion': 'CEPII GeoDist / Language',                        'metodo': 'CSV estatico'},
    'Hofstede':    {'descripcion': 'Hofstede Insights cultural dimensions',           'metodo': 'CSV estatico'},
    'Migracion':   {'descripcion': 'Migracion Colombia - registros de salidas',       'metodo': 'CSV / archivo'},
}
fuentes['Descripcion'] = fuentes['Fuente'].map(lambda f: fuentes_info.get(f, {}).get('descripcion', ''))
fuentes['Metodo de extraccion'] = fuentes['Fuente'].map(lambda f: fuentes_info.get(f, {}).get('metodo', ''))
fuentes = fuentes.sort_values('n_variables', ascending=False).reset_index(drop=True)
fuentes = fuentes[['Fuente', 'n_variables', 'Descripcion', 'Metodo de extraccion']]

print('🌐 Fuentes de datos del proyecto:')
style_table(fuentes, gradient_cols=['n_variables'], gradient_cmap=cmap_custom, format_dict={'n_variables': '{:.0f}'})


🌐 Fuentes de datos del proyecto:


,Fuente,n_variables,Descripcion,Metodo de extraccion
0,World Bank,34,World Development Indicators - API publica REST,wbgapi (DB=2)
1,Hofstede,6,Hofstede Insights cultural dimensions,CSV estatico
2,CEPII,2,CEPII GeoDist / Language,CSV estatico
3,Damodaran,1,NYU Stern Country Risk Premium,requests + Excel
4,Heritage,1,Heritage Foundation Index of Economic Freedom,CSV / API complementaria
5,Migracion,1,Migracion Colombia - registros de salidas,CSV / archivo


## La codificación de relevancia por línea de negocio

El diccionario original del Excel incluye cinco columnas adicionales que no aparecen en la tabla principal de este notebook pero son cruciales para el funcionamiento del sistema: IB, PT, AD, BD y CIB. En cada celda de esas columnas aparece una letra **A** o **B** que codifica la *relevancia de la variable para esa línea de negocio*. La letra **A** indica relevancia **Alta** (la variable es crítica para entender el atractivo del mercado en esa línea, y por tanto su peso relativo dentro de su dimensión es elevado); la letra **B** indica relevancia **Básica** (la variable se incluye en el ranking pero con peso estándar).

Esta codificación es lo que permite que TOPSIS produzca rankings *diferenciados* por línea sin tener que mantener cinco diccionarios paralelos. El procedimiento operativo es el siguiente: durante la construcción del `weight_profile` para cada línea de negocio, las variables marcadas con A reciben un *multiplicador de relevancia* (típicamente alrededor de 1.5) sobre el peso base de la variable dentro de su dimensión, y las marcadas con B mantienen el peso base. Luego se renormalizan los pesos para que sumen uno dentro de la dimensión. El resultado es que en el ranking de Pagos y Flujos (PF), por ejemplo, las variables `personal_remittances_gdp`, `mobile_subscriptions` y `colombian_diaspora_stock` tienen efectivamente más peso que en el ranking de Intermediación Bancaria (IB), donde el peso se concentra en `account_ownership`, `regulatory_quality` y `stock_market_cap_gdp`. La codificación A/B se traduce así en señales sustantivamente distintas para mercados que en términos absolutos podrían parecer similares.

Para visualizar este patrón, vale la pena ver cuántas variables tienen relevancia alta por línea de negocio:

In [8]:
# Relevancia A por linea de negocio (segun el Excel del diccionario)
# Conteo aproximado basado en el listado de variables marcadas como 'A'
relevancia_A = pd.DataFrame({
    'Linea de negocio': ['IB', 'PF', 'AD', 'BD', 'CIB'],
    'Etiqueta': ['Intermediacion Bancaria', 'Pagos y Flujos', 'Activos Digitales',
                 'Bancos Digitales', 'Corporate & Investment Banking'],
    'Variables A (alta)':  [27, 26, 19, 31, 27],
    'Variables B (basica)': [18, 19, 26, 14, 18],
})
relevancia_A['Total'] = relevancia_A['Variables A (alta)'] + relevancia_A['Variables B (basica)']

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Relevancia ALTA (A)', x=relevancia_A['Linea de negocio'],
    y=relevancia_A['Variables A (alta)'],
    marker_color=CIBEST['yellow'],
    text=relevancia_A['Variables A (alta)'], textposition='inside',
    textfont=dict(color=CIBEST['gray'], size=13, family='Arial Black'),
    hovertemplate='<b>%{x}</b><br>Variables A: %{y}<extra></extra>',
))
fig.add_trace(go.Bar(
    name='Relevancia BASICA (B)', x=relevancia_A['Linea de negocio'],
    y=relevancia_A['Variables B (basica)'],
    marker_color=CIBEST['gray_light'],
    text=relevancia_A['Variables B (basica)'], textposition='inside',
    textfont=dict(color='white', size=13),
    hovertemplate='<b>%{x}</b><br>Variables B: %{y}<extra></extra>',
))

fig.update_layout(
    title='<b>Distribución A/B de relevancia por línea de negocio</b><br>'
          '<sub>Cada línea usa las 45 variables, pero asigna más peso a las marcadas con A</sub>',
    barmode='stack',
    xaxis=dict(title='Línea de negocio'),
    yaxis=dict(title='Número de variables', range=[0, 40], gridcolor=CIBEST['gray_border']),
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=420, width=850,
    legend=dict(orientation='h', y=-0.18),
)
fig.show()

print('\n📋 Detalle por linea de negocio:')
style_table(relevancia_A, gradient_cols=['Variables A (alta)'], gradient_cmap=cmap_custom)



📋 Detalle por linea de negocio:


,Linea de negocio,Etiqueta,Variables A (alta),Variables B (basica),Total
0,IB,Intermediacion Bancaria,27,18,45
1,PF,Pagos y Flujos,26,19,45
2,AD,Activos Digitales,19,26,45
3,BD,Bancos Digitales,31,14,45
4,CIB,Corporate & Investment Banking,27,18,45


La tabla muestra una distribución relativamente homogénea en términos de volumen total, ya que **todas las líneas de negocio cuentan con 45 variables evaluadas**. Sin embargo, la composición entre variables de relevancia alta y básica sí evidencia diferencias importantes. La línea **BD (Bancos Digitales) es la que concentra el mayor número de variables A —31 de 45—**, lo que sugiere que este negocio requiere una lectura más fina y prioritaria de múltiples dimensiones del entorno país. Esto es consistente con su naturaleza digital, escalable y altamente dependiente de condiciones como infraestructura tecnológica, adopción financiera, estabilidad regulatoria, madurez del ecosistema digital y confianza institucional.

En un segundo nivel aparecen **IB (Intermediación Bancaria)** y **CIB (Corporate & Investment Banking)**, ambas con **27 variables de relevancia alta y 18 básicas**. Esta composición refleja que, aunque pertenecen a negocios financieros más tradicionales o institucionales, siguen dependiendo de un conjunto amplio de factores críticos para evaluar oportunidades de internacionalización, profundidad de mercado, riesgo país, solidez económica y condiciones regulatorias. Por su parte, **PF (Pagos y Flujos)** presenta una distribución muy cercana, con **26 variables A y 19 B**, lo que indica una necesidad importante de análisis, aunque ligeramente menos intensiva frente a IB y CIB.

La línea con menor concentración de variables de relevancia alta es **AD (Activos Digitales)**, con **19 variables A y 26 variables B**. Esto no implica menor importancia estratégica, sino una priorización más selectiva de los factores críticos para este negocio. En este caso, la evaluación parece enfocarse en un conjunto más acotado de variables altamente determinantes, posiblemente asociadas al entorno regulatorio, la infraestructura digital, la adopción tecnológica y la apertura del mercado a modelos financieros emergentes. En síntesis, la tabla sugiere que **Bancos Digitales demanda el análisis más amplio y sensible**, mientras que **Activos Digitales presenta una estructura más focalizada**, y las demás líneas mantienen un balance intermedio-alto en la cantidad de variables críticas para la toma de decisiones.


Una variable que aparece con relevancia A en **las cinco líneas simultáneamente** es `country_risk_premium` (Damodaran). Esa universalidad no es casual: la prima de riesgo país es una métrica que afecta directamente la rentabilidad esperada de cualquier negocio financiero internacional, sin importar su tipo, porque determina el costo de capital marginal de operar en ese mercado. Por eso es la única variable que el sistema trata como críticamente importante para todas las decisiones de internacionalización.

## Cinco países sintéticos para los ejemplos del notebook

A lo largo del resto del notebook vamos a usar un conjunto pequeño de cinco países sintéticos para que los ejemplos numéricos se puedan seguir con calculadora si se quiere. Los cinco son **Colombia, México, Chile, Panamá y España**, escogidos porque cubren regiones distintas (Sudamérica, Norteamérica, Centroamérica y Europa estratégica) y porque tienen perfiles muy diferentes entre sí, lo que hace que los rankings resultantes sean interesantes pedagógicamente.

Para mantener el ejemplo manejable, usamos solamente **siete variables** representativas de las cinco dimensiones, en lugar de las 35 del sistema completo. Los valores son aproximaciones realistas pero no datos oficiales: el objetivo es que el lector entienda el flujo de cálculo, no que extraiga conclusiones de negocio de este ejemplo reducido. Cuando se ejecuta el sistema real con las 35 variables y los 30 países del alcance, el resultado tiene magnitud distinta pero la lógica es exactamente la misma que veremos en los próximos capítulos.

## Cinco países sintéticos para los ejemplos del notebook

A lo largo del resto del notebook vamos a usar un conjunto pequeño de cinco países sintéticos para que los ejemplos numéricos se puedan seguir con calculadora si se quiere. Los cinco son **Colombia, México, Chile, Panamá y España**, escogidos porque cubren regiones distintas (Sudamérica, Norteamérica, Centroamérica y Europa estratégica) y porque tienen perfiles muy diferentes entre sí, lo que hace que los rankings resultantes sean interesantes pedagógicamente.

Para mantener el ejemplo manejable, usamos solamente **siete variables** que representan a las cinco dimensiones, en lugar de las ~35 del sistema completo. Los valores son aproximaciones realistas pero no datos oficiales: el objetivo es que el lector entienda el flujo de cálculo, no que extraiga conclusiones de negocio de este ejemplo reducido.

In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from src.utils import load_all_configs, resolve_data_path

configs = load_all_configs()
raw_dir = resolve_data_path(configs['settings']['data']['raw_path'])
candidates = sorted(raw_dir.glob('master_raw_*.parquet'), reverse=True)
if not candidates:
    raise FileNotFoundError('Ejecute primero el pipeline de extraccion')
master = pd.read_parquet(candidates[0])

master_wide = (
    master
    .sort_values("year")
    .drop_duplicates(subset=["country_iso3", "variable"], keep="last")
    .pivot(index="country_iso3", columns="variable", values="value")
    .reset_index()
)

#print(master_wide.columns.tolist())
#print(f"Dimensiones en la matriz de decisión: {master_wide.shape[1] - 1} variables (sin contar el ISO3)")

numeric_cols = master_wide.select_dtypes(include="number").columns.tolist()
#print(f"Variables numéricas disponibles para modelado: {len(numeric_cols)}")


# Matriz de decision sintetica para los ejemplos del notebook
cols = [
    "country_iso3",
    "gdp_per_capita_ppp",
    "inflation_rate",
    "financial_system_deposits_gdp",
    "regulatory_quality",
    "internet_users_pct",
    "geographic_distance_km",
    "common_language_spanish",
]

ejemplo = (
    master_wide[cols]
    .dropna()
    .head(5)
    .set_index("country_iso3")
)

paises_demo = ["COL", "MEX", "CHL", "PAN", "ESP"]

ejemplo = (
    master_wide[
        master_wide["country_iso3"].isin(paises_demo)
    ][cols].dropna().set_index("country_iso3")
)


# Direccion de cada variable
direccion = {
    'gdp_per_capita_ppp':      'positive',
    'inflation_rate':          'negative',
    'financial_system_deposits_gdp': 'positive',
    'regulatory_quality':      'positive',
    'internet_users_pct':      'positive',
    'geographic_distance_km':  'negative',
    'common_language_spanish': 'positive',
}

# Dimension de cada variable
dimension_var = {
    'gdp_per_capita_ppp':      'Macro',
    'inflation_rate':          'Macro',
    'financial_system_deposits_gdp': 'Financiera',
    'regulatory_quality':      'Institucional',
    'internet_users_pct':      'Digital',
    'geographic_distance_km':  'Proximidad',
    'common_language_spanish': 'Proximidad',
}



print('Matriz de decisión sintética para el resto del notebook:')
# style_table(master_wide, gradient_cols=numeric_cols, gradient_cmap=cmap_custom, 
#             format_dict={col: '{:.2f}' for col in numeric_cols})


numeric_cols_ejemplo = ejemplo.select_dtypes(include="number").columns.tolist()

style_table(
    ejemplo,
    gradient_cols=numeric_cols_ejemplo,
    gradient_cmap=cmap_custom,
    format_dict={col: '{:.2f}' for col in numeric_cols_ejemplo}
).set_caption("Matriz de decisión sintética (ejemplo)")



KeyError: "['gdp_per_capita_ppp', 'inflation_rate', 'financial_system_deposits_gdp', 'regulatory_quality', 'internet_users_pct'] not in index"

In [ ]:
# ---------------------------------------------------------------------------
# Cargar data real completa y construir matriz de decisión RADAR
# ---------------------------------------------------------------------------

import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np

# Asegurar acceso al proyecto desde notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.utils import (
    load_all_configs,
    resolve_data_path,
    get_variable_catalog,
)

from src.scoring.hybrid_scorer import prepare_decision_matrix


# ---------------------------------------------------------------------------
# 1. Cargar configuración y catálogo real
# ---------------------------------------------------------------------------

configs = load_all_configs()
catalog = get_variable_catalog(configs["variables"])


# ---------------------------------------------------------------------------
# 2. Cargar el master_raw_YYYYMMDD.parquet más reciente
# ---------------------------------------------------------------------------

raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])

pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

candidates = sorted(
    [
        path for path in raw_dir.glob("master_raw_*.parquet")
        if pattern.match(path.name)
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise FileNotFoundError(
        "No se encontró master_raw_YYYYMMDD.parquet. "
        "Ejecute primero el pipeline de extracción."
    )

master_path = candidates[0]
master = pd.read_parquet(master_path)

print(f"Archivo master cargado: {master_path.name}")
print(f"Master cargado: {master.shape}")
print(f"Países únicos: {master['country_iso3'].nunique()}")
print(f"Variables únicas: {master['variable'].nunique()}")


# ---------------------------------------------------------------------------
# 3. Validaciones defensivas del master
# ---------------------------------------------------------------------------

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)

if missing_cols:
    raise ValueError(
        f"Master inválido. Faltan columnas requeridas: {sorted(missing_cols)}"
    )

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

# Limpieza defensiva por compatibilidad con versiones anteriores
master = master[master["variable"] != "wgi_composite"].copy()

if "gdp_growth" not in master["variable"].unique():
    raise ValueError(
        "El master cargado no contiene gdp_growth. "
        "No será posible calcular correctamente el componente Trend."
    )


# ---------------------------------------------------------------------------
# 4. Construir matriz real completa para scoring
# ---------------------------------------------------------------------------

wide_raw, decision_matrix, excluded_countries = prepare_decision_matrix(
    df_long=master,
    configs=configs,
)

print("\nMatriz construida correctamente")
print(f"wide_raw shape: {wide_raw.shape}")
print(f"decision_matrix shape: {decision_matrix.shape}")
print(f"Países excluidos por cobertura/calidad: {excluded_countries}")

print("\nValidación gdp_growth")
print("gdp_growth en master:", "gdp_growth" in master["variable"].unique())
print("gdp_growth en wide_raw:", "gdp_growth" in wide_raw.columns)
print("gdp_growth en decision_matrix:", "gdp_growth" in decision_matrix.columns)

if "gdp_growth" in decision_matrix.columns:
    raise ValueError(
        "gdp_growth está entrando a decision_matrix. "
        "Debe excluirse de TOPSIS y usarse solo en Trend."
    )


# ---------------------------------------------------------------------------
# 5. Construir metadata real de variables usadas en TOPSIS
# ---------------------------------------------------------------------------

metadata_rows = []

for variable in decision_matrix.columns:
    meta = catalog.get(variable, {})

    metadata_rows.append(
        {
            "variable": variable,
            "dimension": meta.get("dimension", "sin_dimension"),
            "source": meta.get("source", "sin_fuente"),
            "direction": meta.get("direction", "sin_direccion"),
            "frequency": meta.get("frequency", "sin_frecuencia"),
            "include_in_topsis": meta.get("include_in_topsis", True),
        }
    )

decision_metadata = pd.DataFrame(metadata_rows)

print("\nVariables reales usadas en decision_matrix:")
display(
    decision_metadata
    .sort_values(["dimension", "variable"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# 6. Matriz real completa para explicación metodológica
# ---------------------------------------------------------------------------

numeric_cols_real = decision_matrix.select_dtypes(include="number").columns.tolist()

print("\nMatriz de decisión real completa")
print(f"Países en decision_matrix: {decision_matrix.shape[0]}")
print(f"Variables TOPSIS en decision_matrix: {decision_matrix.shape[1]}")

# Si existe style_table en el notebook, usarlo.
# Si no existe, mostrar DataFrame normal.
try:
    display(
        style_table(
            decision_matrix.reset_index(),
            gradient_cols=numeric_cols_real,
            gradient_cmap=cmap_custom,
            format_dict={col: "{:.3f}" for col in numeric_cols_real},
        ).set_caption("Matriz de decisión real completa — normalizada y orientada")
    )
except NameError:
    display(decision_matrix.reset_index().round(3))

Mirando esta tabla de cerca ya se intuye el problema central que el sistema tiene que resolver: **ningún país domina a los demás en todas las variables**. España tiene el PIB per cápita más alto y la mejor digitalización, pero está muy lejos de Bogotá. Panamá está cerquita y comparte idioma, pero su mercado es pequeño. Colombia tiene la inflación más alta y un PIB per cápita bajo, pero la distancia es cero. Chile tiene la mejor calidad regulatoria pero está lejos. Esa pluralidad de fortalezas y debilidades es exactamente lo que hace que las técnicas multicriterio sean necesarias: no hay una respuesta obvia.

---

# Capítulo 4 — BWM: ¿qué tanto importa cada dimensión?

## La intuición

Antes de poder rankear países necesitamos saber *cuánto pesa cada dimensión en la decisión*. Una persona del Comité podría pensar que la dimensión Institucional es el doble de importante que la Digital, y otra persona podría pensar lo contrario. El sistema necesita un mecanismo para que la alta dirección exprese esas preferencias de forma estructurada, y luego convertir esas preferencias en pesos numéricos que se puedan usar en el algoritmo de ranking.

La técnica que RADAR Cibest usa para esto se llama [**BWM: Best-Worst Method**](https://bestworstmethod.com/), propuesta por Jafar Rezaei en 2015. Su gran ventaja sobre el método clásico AHP de Saaty es que requiere **muchas menos comparaciones** del experto. AHP pide $n(n-1)/2$ comparaciones, lo que para cinco dimensiones son diez. BWM pide solo $2n-3$ comparaciones, lo que para cinco dimensiones son siete. Esa reducción no es trivial: significa que un ejecutivo puede completar la elicitación BWM en una sesión de 90 minutos en lugar de necesitar varias sesiones, y los estudios muestran que los pesos resultantes son **más consistentes** estadísticamente que los del AHP.

## Cómo funciona BWM, paso a paso

El procedimiento [BWM](https://bestworstmethod.com/wp-content/uploads/2020/01/Best-Worst-Method-BWM-2019.pdf) tiene cuatro pasos muy concretos que el ejecutivo recorre con un facilitador. **Primero**, el ejecutivo identifica cuál dimensión considera *la más importante* (la "Best") y cuál considera *la menos importante* (la "Worst"). Esa elección no tiene que ser racional ni justificada con datos; refleja la intuición estratégica del ejecutivo y eso es exactamente lo que queremos capturar.

**Segundo**, el ejecutivo compara la dimensión Best con todas las demás, usando una escala entera de 1 a 9 donde 1 significa "igualmente importantes" y 9 significa "la Best es extremadamente más importante que esta otra". Estas comparaciones forman el llamado *vector Best-to-Others*. **Tercero**, hace lo simétrico: compara cada una de las demás dimensiones contra la Worst, formando el *vector Others-to-Worst*. **Cuarto**, los dos vectores entran a un modelo de optimización que produce los pesos finales.

## La fórmula del modelo de optimización

El modelo BWM se formula así: dado que $a_{Bj}$ es la comparación entre la mejor dimensión y la dimensión $j$, y $a_{jW}$ es la comparación entre la dimensión $j$ y la peor, los pesos $w_j$ se obtienen resolviendo:

$$
\min \; \xi
$$

$$
\text{sujeto a:} \quad |w_B - a_{Bj} \cdot w_j| \leq \xi \quad \forall j
$$

$$
|w_j - a_{jW} \cdot w_W| \leq \xi \quad \forall j
$$

$$
\sum_{j} w_j = 1, \quad w_j \geq 0
$$

La variable $\xi$ (xi) mide el *grado de inconsistencia* en los juicios del experto: si el experto fuera perfectamente consistente, sería posible encontrar pesos que satisfacen todas las restricciones con $\xi = 0$. En la práctica $\xi$ termina siendo positivo, y entre más pequeño, más consistente fue el experto. El cociente $\xi^* / CI$ (donde $CI$ es el *Consistency Index* tabulado por Rezaei) se llama *Consistency Ratio* y debe estar por debajo de 0.10 para que los pesos se consideren confiables.

## Ejemplo numérico completo

Vamos a simular la elicitación BWM con un solo ejecutivo, que considera que la dimensión Institucional es la *más importante* y la dimensión Digital-Tecnológica es la *menos importante*. Vamos a producir los pesos y validar su consistencia.

In [ ]:
# Juicios BWM simulados de un ejecutivo de Cibest
juicios_ejecutivo = {
    'criteria': ['macro', 'financial', 'institutional', 'digital_tech', 'proximity'],
    'best': 'institutional',         # la dimension MAS importante
    'worst': 'digital_tech',         # la dimension MENOS importante
    'best_to_others': {              # cuanto mas importante es la 'best' vs cada otra (1-9)
        'macro':          3,
        'financial':      2,
        'institutional':  1,         # comparacion consigo misma = 1
        'digital_tech':   7,
        'proximity':      4,
    },
    'others_to_worst': {             # cuanto mas importante es cada otra vs la 'worst' (1-9)
        'macro':          5,
        'financial':      6,
        'institutional':  7,
        'digital_tech':   1,         # comparacion consigo misma = 1
        'proximity':      4,
    },
}

# Mostrar los juicios como tabla
df_juicios = pd.DataFrame({
    'Dimensión': juicios_ejecutivo['criteria'],
    'Best-to-Others (vs Institucional)': [juicios_ejecutivo['best_to_others'][c] for c in juicios_ejecutivo['criteria']],
    'Others-to-Worst (vs Digital)':      [juicios_ejecutivo['others_to_worst'][c] for c in juicios_ejecutivo['criteria']],
})
print('Juicios del ejecutivo:')
print(f"   Best = {juicios_ejecutivo['best']}    |    Worst = {juicios_ejecutivo['worst']}\n")
style_table(df_juicios)


Juicios del ejecutivo:
   Best = institutional    |    Worst = digital_tech



,Dimensión,Best-to-Others (vs Institucional),Others-to-Worst (vs Digital)
0,macro,3,5
1,financial,2,6
2,institutional,1,7
3,digital_tech,7,1
4,proximity,4,4


In [ ]:
# Resolucion del modelo BWM con SLSQP
from scipy.optimize import minimize

def resolver_bwm(juicios):
    criteria = juicios['criteria']
    best = juicios['best']
    worst = juicios['worst']
    bto = juicios['best_to_others']
    otw = juicios['others_to_worst']
    n = len(criteria)
    idx = {c: i for i, c in enumerate(criteria)}

    def objective(x): return x[-1]   # minimizar xi (ultimo elemento)

    cons = [{'type': 'eq', 'fun': lambda x: np.sum(x[:n]) - 1.0}]
    for c in criteria:
        if c != best:
            j, b = idx[c], idx[best]
            a = bto[c]
            cons.append({'type': 'ineq', 'fun': lambda x, b=b, j=j, a=a: x[-1] - abs(x[b] - a * x[j])})
    for c in criteria:
        if c != worst:
            j, w = idx[c], idx[worst]
            a = otw[c]
            cons.append({'type': 'ineq', 'fun': lambda x, j=j, w=w, a=a: x[-1] - abs(x[j] - a * x[w])})

    x0 = np.concatenate([np.full(n, 1.0/n), [0.1]])
    bounds = [(0.0, 1.0)] * n + [(0.0, None)]
    res = minimize(objective, x0=x0, method='SLSQP', bounds=bounds,
                   constraints=cons, options={'maxiter': 500, 'ftol': 1e-9})
    pesos = res.x[:n] / res.x[:n].sum()
    xi_star = res.x[-1]
    return {c: float(pesos[idx[c]]) for c in criteria}, float(xi_star)

pesos, xi_star = resolver_bwm(juicios_ejecutivo)

# Tabla CI segun Rezaei (2015)
CI_TABLE = {1: 0.00, 2: 0.44, 3: 1.00, 4: 1.63, 5: 2.30, 6: 3.00, 7: 3.73, 8: 4.47, 9: 5.23}
a_BW = juicios_ejecutivo['best_to_others'][juicios_ejecutivo['worst']]
ci = CI_TABLE[a_BW]
cr = xi_star / ci

print(f'\n Resolución del modelo BWM:')
print(f'   ξ* = {xi_star:.4f}    (inconsistencia)')
print(f'   CI = {ci:.2f}        (a_BW = {a_BW}, según tabla Rezaei 2015)')
print(f'   CR = ξ*/CI = {cr:.4f}    {"✅ CONSISTENTE" if cr < 0.10 else "⚠️  INCONSISTENTE"}\n')

df_pesos = pd.DataFrame({
    'Dimensión': list(pesos.keys()),
    'Peso BWM':  list(pesos.values()),
    'Peso (%)':  [f'{v*100:.1f}%' for v in pesos.values()],
})
style_table(df_pesos, gradient_cols=['Peso BWM'], gradient_cmap=cmap_custom,
            format_dict={'Peso BWM': '{:.4f}'})



 Resolución del modelo BWM:
   ξ* = 0.0775    (inconsistencia)
   CI = 3.73        (a_BW = 7, según tabla Rezaei 2015)
   CR = ξ*/CI = 0.0208    ✅ CONSISTENTE



,Dimensión,Peso BWM,Peso (%)
0,macro,0.1646,16.5%
1,financial,0.2470,24.7%
2,institutional,0.4165,41.6%
3,digital_tech,0.0484,4.8%
4,proximity,0.1235,12.3%


In [ ]:
# Visualizacion interactiva de los pesos BWM
fig = go.Figure()
dims = list(pesos.keys())
vals = [pesos[d] for d in dims]
colors_bar = [CIBEST['yellow'] if d == juicios_ejecutivo['best']
              else CIBEST['red'] if d == juicios_ejecutivo['worst']
              else CIBEST['gray'] for d in dims]

fig.add_trace(go.Bar(
    x=dims, y=vals,
    marker=dict(color=colors_bar, line=dict(color=CIBEST['gray'], width=1.5)),
    text=[f'{v*100:.1f}%' for v in vals],
    textposition='outside',
    textfont=dict(size=13, color=CIBEST['gray']),
    hovertemplate='<b>%{x}</b><br>Peso: %{y:.4f}<br>%{text}<extra></extra>',
))
fig.update_layout(
    title=f'<b>Pesos BWM resultantes</b><br><sub>Best (dorado) = {juicios_ejecutivo["best"]}, '
          f'Worst (rojo) = {juicios_ejecutivo["worst"]} · CR = {cr:.3f}</sub>',
    xaxis_title='Dimensión',
    yaxis_title='Peso',
    yaxis=dict(range=[0, max(vals) * 1.25], gridcolor=CIBEST['gray_border']),
    plot_bgcolor=CIBEST['white'],
    paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=440, showlegend=False,
)
fig.show()


Lo que esta gráfica muestra es la "huella estratégica" del ejecutivo: cuánto pesa cada dimensión según su criterio. La dimensión Best (Institucional, en dorado) se llevó el peso más alto, la dimensión Worst (Digital, en rojo) el más bajo, y las demás quedaron ordenadas en función de las comparaciones intermedias. La suma de todos los pesos es exactamente uno por construcción, lo que permite usarlos directamente como vector de pesos en el algoritmo TOPSIS que viene en el próximo capítulo.

## ¿Y si hay varios ejecutivos en el panel?

En la sesión real con la alta dirección de Cibest no participa un solo ejecutivo, sino un panel. Cada miembro del panel completa su propia elicitación y produce su propio vector de pesos. Para integrar los vectores individuales en un vector consolidado el sistema usa la **media geométrica**, que es el estándar en la literatura de decisión multicriterio porque preserva las razones entre pesos:

$$
w_j^{\text{panel}} = \sqrt[m]{\prod_{k=1}^{m} w_{j,k}}
$$

donde $m$ es el número de ejecutivos en el panel y $w_{j,k}$ es el peso de la dimensión $j$ según el ejecutivo $k$. Después se renormaliza para que la suma sea uno.

## De pesos por dimensión a pesos por variable

Hay un último paso conceptualmente sencillo pero operativamente importante. BWM nos da pesos para las cinco *dimensiones*, pero TOPSIS necesita pesos para las ~35 *variables*. La conversión se hace en dos niveles. Dentro de cada dimensión, se asigna un peso a cada variable (puede ser uniforme, o derivado de otra elicitación específica). Luego el peso final de una variable es el producto:

$$
w_v^{\text{final}} = w_{\text{dim}(v)} \cdot w_v^{\text{dentro de dim}}
$$

Esto garantiza que la suma de todos los pesos de variables sea uno, y que las variables de una dimensión "más importante" tengan más peso total que las de una dimensión "menos importante".

---

# Capítulo 5 — TOPSIS: ranking de los países

## La intuición geométrica

Una vez que tenemos los pesos de las variables (gracias a BWM), necesitamos un método que nos diga qué tan atractivo es cada país. La técnica que usa RADAR Cibest se llama [**TOPSIS** (Technique for Order of Preference by Similarity to Ideal Solution)](https://www.youtube.com/watch?v=kfcN7MuYVeI), propuesta por Hwang y Yoon en 1981, y es probablemente la técnica multicriteria más usada en literatura financiera y bancaria.

La intuición de [TOPSIS](https://en.wikipedia.org/wiki/TOPSIS) es geométrica y muy hermosa. Imagine usted que cada país es un punto en un espacio donde cada eje es una variable (PIB, inflación, calidad regulatoria, etc.). En ese espacio existen dos puntos especiales: la **solución ideal positiva**, que sería un país hipotético que tiene el mejor valor en todas las variables a la vez (el PIB más alto, la inflación más baja, la mejor calidad regulatoria, etc.); y la **solución ideal negativa**, que sería el opuesto: peor valor en todo. Ningún país real va a ser ni el ideal positivo ni el ideal negativo, pero podemos medir, para cada país, qué tan cerca está del ideal positivo y qué tan cerca está del ideal negativo.

Un país es atractivo si está *cerca* del ideal positivo *y a la vez lejos* del ideal negativo. TOPSIS combina esas dos distancias en una sola métrica llamada *closeness coefficient* o coeficiente de cercanía, que está siempre entre 0 y 1 y que define el ranking final.

## El algoritmo completo, paso a paso

El procedimiento TOPSIS tiene seis pasos. Vamos a recorrerlos uno por uno con la matriz de cinco países y siete variables que definimos en el capítulo tres.

### Paso 1: normalización de la matriz y aplicación de dirección

Antes de cualquier cosa hay que poner todas las variables en la misma escala. Las unidades del PIB (USD), de la inflación (%) y de la distancia (km) son incomparables. La normalización **min-max** convierte cada variable a una escala de 0 a 1:

$$
R_{ij} = \frac{x_{ij} - \min_i(x_{ij})}{\max_i(x_{ij}) - \min_i(x_{ij})}
$$

Luego se aplica la **dirección**: si una variable tiene dirección negativa (como la inflación), se invierte para que "alto siga significando bueno":

$$
R_{ij}^{\text{orientado}} = 1 - R_{ij} \quad \text{si } \text{dirección}(j) = \text{negativa}
$$

In [ ]:
# Paso 1: normalizacion min-max + aplicacion de direccion
def normalizar_y_orientar(matriz, direccion):
    normalizada = pd.DataFrame(index=matriz.index, columns=matriz.columns, dtype=float)
    for col in matriz.columns:
        vmin, vmax = matriz[col].min(), matriz[col].max()
        if vmax - vmin == 0:
            normalizada[col] = 0.5
        else:
            normalizada[col] = (matriz[col] - vmin) / (vmax - vmin)
        if direccion[col] == 'negative':
            normalizada[col] = 1.0 - normalizada[col]
    return normalizada

matriz_norm = normalizar_y_orientar(ejemplo, direccion)
print('Matriz normalizada y orientada (todas en [0,1], mayor = mejor):')
style_table(matriz_norm.round(3),
            gradient_cols=matriz_norm.columns.tolist(), gradient_cmap=cmap_custom,
            format_dict={c: '{:.3f}' for c in matriz_norm.columns})


Matriz normalizada y orientada (todas en [0,1], mayor = mejor):


variable,gdp_per_capita_ppp,inflation_rate,financial_system_deposits_gdp,regulatory_quality,internet_users_pct,geographic_distance_km,common_language_spanish
country_iso3,,,,,,,
CHL,0.388,0.391,0.356,1.000,0.993,0.471,0.500
COL,0.000,0.000,0.000,0.214,0.286,1.000,0.500
ESP,1.000,0.648,1.000,0.836,1.000,0.000,0.500
MEX,0.108,0.319,0.045,0.000,0.450,0.565,0.500
PAN,0.534,1.000,0.297,0.125,0.000,0.899,0.500


Note que ahora todos los valores están entre 0 y 1, y que la columna `inflacion` cambió de sentido: Colombia que tenía 6.6% (la peor) ahora tiene valor 0 en la versión normalizada y orientada (porque su inflación es la peor para el atractivo), y Panama que tenía 0.69% (la mejor) ahora tiene 1.

### Paso 2: aplicación de pesos

Cada columna se multiplica por su peso. Para el ejemplo vamos a usar pesos uniformes (cada variable pesa $1/7$), lo cual es una simplificación frente al sistema real donde los pesos vienen de BWM. Esto produce la *matriz de decisión ponderada* $V$:

$$
V_{ij} = w_j \cdot R_{ij}^{\text{orientado}}
$$

In [ ]:
# Paso 2: ponderacion
pesos_uniformes = {c: 1/len(matriz_norm.columns) for c in matriz_norm.columns}
matriz_ponderada = matriz_norm.copy()
for col in matriz_ponderada.columns:
    matriz_ponderada[col] = matriz_ponderada[col] * pesos_uniformes[col]

print(f'Matriz ponderada (peso uniforme = {1/7:.4f} por variable):')
style_table(matriz_ponderada.round(4),
            gradient_cols=matriz_ponderada.columns.tolist(), gradient_cmap=cmap_custom,
            format_dict={c: '{:.4f}' for c in matriz_ponderada.columns})


Matriz ponderada (peso uniforme = 0.1429 por variable):


variable,gdp_per_capita_ppp,inflation_rate,financial_system_deposits_gdp,regulatory_quality,internet_users_pct,geographic_distance_km,common_language_spanish
country_iso3,,,,,,,
CHL,0.0555,0.0558,0.0509,0.1429,0.1418,0.0672,0.0714
COL,0.0000,0.0000,0.0000,0.0306,0.0409,0.1429,0.0714
ESP,0.1429,0.0926,0.1429,0.1195,0.1429,0.0000,0.0714
MEX,0.0154,0.0456,0.0064,0.0000,0.0643,0.0808,0.0714
PAN,0.0763,0.1429,0.0424,0.0178,0.0000,0.1284,0.0714


### Paso 3: cálculo de soluciones ideal positiva e ideal negativa

La solución ideal positiva $A^+$ toma el máximo de cada columna; la ideal negativa $A^-$ toma el mínimo:

$$
A^+ = \langle \max_i V_{i1}, \max_i V_{i2}, \dots, \max_i V_{ij} \rangle
$$

$$
A^- = \langle \min_i V_{i1}, \min_i V_{i2}, \dots, \min_i V_{ij} \rangle
$$

In [ ]:
# Paso 3: soluciones ideales
A_pos = matriz_ponderada.max()
A_neg = matriz_ponderada.min()

ideales = pd.DataFrame({'A⁺ (ideal positivo)': A_pos.round(4),
                        'A⁻ (ideal negativo)': A_neg.round(4)})
print('Soluciones ideales:')
style_table(ideales, format_dict={c: '{:.4f}' for c in ideales.columns})


Soluciones ideales:


,A⁺ (ideal positivo),A⁻ (ideal negativo)
variable,,
gdp_per_capita_ppp,0.1429,0.0000
inflation_rate,0.1429,0.0000
financial_system_deposits_gdp,0.1429,0.0000
regulatory_quality,0.1429,0.0000
internet_users_pct,0.1429,0.0000
geographic_distance_km,0.1429,0.0000
common_language_spanish,0.0714,0.0714


### Paso 4: distancias euclidianas a las soluciones ideales

Para cada país $i$ se calcula su distancia al ideal positivo y al ideal negativo:

$$
d_i^+ = \sqrt{\sum_{j} (V_{ij} - A_j^+)^2}, \quad d_i^- = \sqrt{\sum_{j} (V_{ij} - A_j^-)^2}
$$

In [ ]:
# Paso 4: distancias euclidianas
d_pos = np.sqrt(((matriz_ponderada - A_pos) ** 2).sum(axis=1))
d_neg = np.sqrt(((matriz_ponderada - A_neg) ** 2).sum(axis=1))

distancias = pd.DataFrame({'d⁺ (distancia al ideal positivo)': d_pos.round(4),
                           'd⁻ (distancia al ideal negativo)': d_neg.round(4)})
print('Distancias euclidianas:')
style_table(distancias, format_dict={c: '{:.4f}' for c in distancias.columns})


Distancias euclidianas:


,d⁺ (distancia al ideal positivo),d⁻ (distancia al ideal negativo)
country_iso3,,
CHL,0.1714,0.2320
COL,0.2902,0.1517
ESP,0.1532,0.2900
MEX,0.2734,0.1141
PAN,0.2253,0.2118


### Paso 5: cálculo del coeficiente de cercanía

El coeficiente $C^*$ combina ambas distancias:

$$
C_i^* = \frac{d_i^-}{d_i^+ + d_i^-}
$$

La interpretación es elegante: $C^*$ vale 1 si el país coincide con el ideal positivo, 0 si coincide con el ideal negativo, y valores intermedios en proporción a qué tan cerca está de uno u otro. **Mayor $C^*$ significa mayor atractivo del mercado**.

### Paso 6: ranking final

Se ordenan los países por $C^*$ descendente:

In [ ]:
# Pasos 5 y 6: coeficiente de cercania y ranking
cc = d_neg / (d_pos + d_neg)
ranking_final = pd.DataFrame({
    'd⁺':     d_pos.round(4),
    'd⁻':     d_neg.round(4),
    'C*':     cc.round(4),
}).sort_values('C*', ascending=False)
ranking_final.insert(0, 'Rank', range(1, len(ranking_final) + 1))

print('Ranking final TOPSIS:')
style_table(ranking_final, gradient_cols=['C*'], gradient_cmap=cmap_custom,
            format_dict={'d⁺': '{:.4f}', 'd⁻': '{:.4f}', 'C*': '{:.4f}', 'Rank': '{:.0f}'})


Ranking final TOPSIS:


,Rank,d⁺,d⁻,C*
country_iso3,,,,
ESP,1,0.1532,0.2900,0.6542
CHL,2,0.1714,0.2320,0.5751
PAN,3,0.2253,0.2118,0.4845
COL,4,0.2902,0.1517,0.3433
MEX,5,0.2734,0.1141,0.2944


In [ ]:
# Visualizacion interactiva del ranking TOPSIS
fig = go.Figure()
df_plot = ranking_final.reset_index().sort_values('C*', ascending=True)
fig.add_trace(go.Bar(
    x=df_plot['C*'], y=df_plot.iloc[:, 0],  # primera columna = pais (index reseteado)
    orientation='h',
    marker=dict(color=df_plot['C*'], colorscale=[[0, CIBEST['red']],
                                                   [0.5, CIBEST['gold']],
                                                   [1.0, CIBEST['green']]],
                line=dict(color=CIBEST['gray'], width=1)),
    text=df_plot['C*'].round(3),
    textposition='outside',
    textfont=dict(color=CIBEST['gray'], size=12),
    hovertemplate='<b>%{y}</b><br>C* = %{x:.4f}<extra></extra>',
))
fig.update_layout(
    title='<b>Ranking TOPSIS — Coeficiente de cercanía C*</b><br>'
          '<sub>Países ordenados por proximidad al ideal positivo</sub>',
    xaxis_title='C* (coeficiente de cercanía)',
    yaxis_title='País',
    xaxis=dict(range=[0, 1.0], gridcolor=CIBEST['gray_border']),
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=400, showlegend=False, margin=dict(l=100),
)
fig.show()


## Una visualización geométrica para fijar la intuición

Para ver geométricamente lo que TOPSIS está haciendo, podemos proyectar el problema a dos dimensiones. Si tomamos solo dos variables (por ejemplo PIB per cápita y Calidad Regulatoria), podemos dibujar cada país como un punto en el plano y marcar dónde están las soluciones ideales:

In [ ]:
# Proyeccion 2D para intuicion geometrica de TOPSIS
fig = go.Figure()

# Solo dos variables para visualizacion 2D
xs = matriz_norm['gdp_per_capita_ppp']
ys = matriz_norm['regulatory_quality']

# Marcadores de cada pais
fig.add_trace(go.Scatter(
    x=xs, y=ys, mode='markers+text',
    marker=dict(size=22, color=CIBEST['gray'], line=dict(color=CIBEST['gold'], width=2)),
    text=xs.index, textposition='top center',
    textfont=dict(size=12, color=CIBEST['gray']),
    name='Países', hovertemplate='<b>%{text}</b><br>PIB PPP norm.: %{x:.3f}<br>Calidad reg. norm.: %{y:.3f}<extra></extra>',
))

# Ideal positivo (en la esquina superior derecha del cuadrado [0,1]x[0,1])
fig.add_trace(go.Scatter(
    x=[xs.max()], y=[ys.max()], mode='markers+text',
    marker=dict(size=28, color=CIBEST['green'], symbol='star', line=dict(color=CIBEST['gray'], width=2)),
    text=['A⁺'], textposition='top center', textfont=dict(size=15, color=CIBEST['green']),
    name='Ideal positivo (A⁺)',
))

# Ideal negativo (esquina inferior izquierda)
fig.add_trace(go.Scatter(
    x=[xs.min()], y=[ys.min()], mode='markers+text',
    marker=dict(size=28, color=CIBEST['red'], symbol='star', line=dict(color=CIBEST['gray'], width=2)),
    text=['A⁻'], textposition='bottom center', textfont=dict(size=15, color=CIBEST['red']),
    name='Ideal negativo (A⁻)',
))

# Lineas hacia los ideales para el primer pais del ranking
top_country = ranking_final.index[0]
fig.add_trace(go.Scatter(
    x=[xs[top_country], xs.max()], y=[ys[top_country], ys.max()],
    mode='lines', line=dict(color=CIBEST['green'], width=2, dash='dash'),
    name=f'd⁺ desde {top_country}', showlegend=True,
    hoverinfo='skip',
))
fig.add_trace(go.Scatter(
    x=[xs[top_country], xs.min()], y=[ys[top_country], ys.min()],
    mode='lines', line=dict(color=CIBEST['red'], width=2, dash='dash'),
    name=f'd⁻ desde {top_country}', showlegend=True,
    hoverinfo='skip',
))

fig.update_layout(
    title='<b>Visualización geométrica de TOPSIS</b><br>'
          '<sub>Proyección 2D: PIB per cápita PPP vs. Calidad regulatoria (normalizadas)</sub>',
    xaxis=dict(title='PIB per cápita PPP (normalizado)', range=[-0.05, 1.1], gridcolor=CIBEST['gray_border']),
    yaxis=dict(title='Calidad regulatoria (normalizada)', range=[-0.05, 1.1], gridcolor=CIBEST['gray_border']),
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=550, legend=dict(orientation='h', y=-0.15),
)
fig.show()


La figura ilustra perfectamente la intuición de TOPSIS. Los cinco países están en distintos puntos del cuadrado unitario. La estrella verde arriba a la derecha es el ideal positivo (mejor en todo), la estrella roja abajo a la izquierda es el ideal negativo (peor en todo). Las líneas punteadas muestran las dos distancias del país top: a más corta sea la distancia verde (al ideal positivo) y a más larga sea la distancia roja (al ideal negativo), mejor el ranking. En el sistema real esto sucede no en dos dimensiones sino en ~35 dimensiones, pero la idea es la misma.

## Scores parciales por dimensión: una sutileza importante

Hay un detalle del TOPSIS implementado en RADAR Cibest que merece mencionarse explícitamente. Además del coeficiente $C^*$ global, el sistema ejecuta TOPSIS también sobre cada **subconjunto** de variables que pertenece a una misma dimensión, generando así *scores parciales por dimensión*. Estos scores parciales son los que alimentan el motor de señales y el perfil de fortalezas/debilidades de cada país. Sin esos scores parciales el sistema no podría decir cosas como "Panamá es fuerte en proximidad pero débil en macro": solo podría decir "Panamá tiene C* = 0.6 globalmente".

## Múltiples ejecuciones por línea de negocio

La otra sutileza importante es que TOPSIS se ejecuta **seis veces en total** en el sistema real: una vez con pesos globales (BWM agregado del panel) y cinco veces más, una por cada línea de negocio, donde los pesos cambian según el `weight_profile` de la línea. Esto es lo que produce los rankings diferenciados por línea de los que hablábamos al inicio. Computacionalmente es trivial; conceptualmente es la respuesta a la pregunta "¿cuáles son los mercados más atractivos *para esta línea en particular*?".

---

# Capítulo 6 — Modelo gravitacional: el factor de proximidad con Colombia

## La intuición física

Hay una observación empírica muy robusta en economía internacional: **los flujos de bienes, capital, personas y servicios entre dos países disminuyen con la distancia entre ellos y aumentan con sus tamaños económicos**. La fórmula que captura esta regularidad se llama, por analogía con la ley de gravitación universal de Newton, *modelo gravitacional*. En la versión clásica de Anderson y van Wincoop (2003), el flujo bilateral entre dos países $i$ y $j$ tiene la forma:

$$
F_{ij} \propto \frac{Y_i^{\alpha} \cdot Y_j^{\beta}}{D_{ij}^{\gamma}}
$$

donde $Y_i$ e $Y_j$ son los PIB de los dos países y $D_{ij}$ es la distancia entre ellos. Los exponentes $\alpha$, $\beta$ y $\gamma$ se estiman empíricamente y suelen estar cerca de 1.

En el contexto bancario específicamente, Brei y von Peter (2018) muestran que duplicar la distancia entre dos países reduce los flujos bancarios bilaterales entre 30% y 50%. Ese hallazgo es central para RADAR Cibest: la distancia importa muchísimo para el negocio bancario. Pero "distancia" en sentido económico no es solo distancia geográfica; incluye también *distancia cultural*, *distancia institucional* y *distancia lingüística*. Ghemawat (2001) formalizó esto en el llamado *marco CAGE* (Cultural, Administrative, Geographic, Economic).

## El Índice de Proximidad Compuesto (IPC)

RADAR Cibest no estima una regresión gravitacional completa porque no necesita predecir flujos bilaterales en valores; lo que necesita es un **índice resumen** de qué tan "cerca" está cada país de Colombia en sentido amplio. Ese índice se llama **Índice de Proximidad Compuesto (IPC)** y se construye así. Para cada componente $k$ de proximidad (distancia geográfica, distancia cultural Hofstede, idioma compartido, comercio bilateral, stock de diáspora), se normaliza el componente a la escala [0,1] aplicando su dirección, y luego se hace una combinación lineal:

$$
\text{IPC}_j = \sum_{k} w_k \cdot \text{prox}_k(\text{COL}, j)
$$

donde $w_k$ son los pesos de cada componente (por defecto equiponderados) y $\text{prox}_k(\text{COL}, j)$ es el componente $k$ normalizado y orientado de modo que mayor valor signifique mayor proximidad. Por construcción Colombia tiene $\text{IPC} = 1$ (es la "máxima proximidad consigo misma").

## Ejemplo numérico con los cinco países

In [ ]:
# Calculo del IPC para los cinco paises del ejemplo
# Usamos las dos variables de proximidad de nuestra matriz: distancia y idioma
componentes_ipc = pd.DataFrame({
    'distancia_km':          ejemplo['geographic_distance_km'],
    'idioma_espanol':        ejemplo['common_language_spanish'],
})

# Normalizar y orientar: distancia (negativa) -> invertir
def norm_y_orientar_componente(serie, direccion):
    vmin, vmax = serie.min(), serie.max()
    if vmax - vmin == 0:
        n = pd.Series(0.5, index=serie.index)
    else:
        n = (serie - vmin) / (vmax - vmin)
    if direccion == 'negative':
        n = 1.0 - n
    return n

prox_distancia = norm_y_orientar_componente(componentes_ipc['distancia_km'], 'negative')
prox_idioma    = norm_y_orientar_componente(componentes_ipc['idioma_espanol'], 'positive')

# IPC equiponderado entre los dos componentes
ipc = 0.5 * prox_distancia + 0.5 * prox_idioma
ipc.loc['COL'] = 1.0   # por definicion

df_ipc = pd.DataFrame({
    'Distancia (km)':                componentes_ipc['distancia_km'].astype(int),
    'Proximidad geográfica (norm.)': prox_distancia.round(3),
    'Idioma español (1/0)':          componentes_ipc['idioma_espanol'].astype(int),
    'IPC':                           ipc.round(3),
}).sort_values('IPC', ascending=False)

print('Índice de Proximidad Compuesto (IPC):')
style_table(df_ipc, gradient_cols=['IPC'], gradient_cmap='YlOrBr',
            format_dict={'IPC': '{:.3f}', 'Proximidad geográfica (norm.)': '{:.3f}'})


Índice de Proximidad Compuesto (IPC):


,Distancia (km),Proximidad geográfica (norm.),Idioma español (1/0),IPC
country_iso3,,,,
COL,0,1.000,1,1.000
PAN,810,0.899,1,0.700
MEX,3490,0.565,1,0.533
CHL,4250,0.471,1,0.485
ESP,8030,0.000,1,0.250


In [ ]:
# Visualizacion del IPC
fig = go.Figure()
df_sorted = df_ipc.sort_values('IPC')
fig.add_trace(go.Bar(
    y=df_sorted.index, x=df_sorted['IPC'],
    orientation='h',
    marker=dict(color=df_sorted['IPC'],
                colorscale=[[0, CIBEST['gray_bg']], [0.5, CIBEST['gray_light']], [1, CIBEST['yellow']]],
                line=dict(color=CIBEST['gray'], width=1)),
    text=df_sorted['IPC'].round(3),
    textposition='outside',
    textfont=dict(color=CIBEST['gray']),
    hovertemplate='<b>%{y}</b><br>IPC = %{x:.3f}<extra></extra>',
))
fig.update_layout(
    title='<b>Índice de Proximidad Compuesto (IPC)</b><br>'
          '<sub>Afinidad de cada país con Colombia · Colombia = 1.0 por construcción</sub>',
    xaxis=dict(title='IPC', range=[0, 1.1], gridcolor=CIBEST['gray_border']),
    yaxis_title='País',
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=400, margin=dict(l=100), showlegend=False,
)
fig.show()


El IPC del ejemplo confirma lo esperado: Panamá está muy cerca de Colombia (corto vuelo, mismo idioma), México intermedio (mismo idioma pero más lejos), y España queda al final pese a compartir idioma porque la distancia geográfica es enorme. En el sistema real el IPC integra más componentes (distancia cultural Hofstede, comercio bilateral, stock de diáspora), lo que afina aún más la medida.

## ¿Por qué incorporar proximidad si TOPSIS ya considera variables de proximidad?

Esta es una pregunta legítima y conviene responderla. TOPSIS efectivamente puede incluir la distancia y el idioma como variables más, y de hecho lo hace en el sistema real. Entonces ¿para qué tener un IPC separado? La respuesta tiene dos partes. La primera es **interpretabilidad**: tener un IPC explícito permite responder al Comité preguntas del tipo "¿qué tanto pesa la proximidad con Colombia en este ranking?" con una cifra concreta ($\beta$ del score compuesto que veremos en el siguiente capítulo). La segunda es **flexibilidad estratégica**: si en algún momento Cibest quiere experimentar con una estrategia "más globalista" (priorizar atractivo absoluto) o "más regionalista" (priorizar proximidad), se puede hacer ajustando $\beta$ sin tocar la estructura de pesos de TOPSIS. Esa flexibilidad es valiosa.

---

# Capítulo 7 — El score RADAR compuesto

## Tres componentes que se integran en uno

Hasta ahora tenemos tres salidas separadas. TOPSIS produce un coeficiente de cercanía $C^*$ para cada país, que mide su atractivo absoluto según las variables y pesos. El modelo gravitacional produce el IPC, que mide la afinidad bilateral con Colombia. Y por separado, el sistema calcula un **factor de tendencia** que captura el dinamismo reciente del mercado (típicamente el promedio del crecimiento real del PIB en los últimos tres años, normalizado).

El score RADAR final integra los tres componentes en una sola cifra mediante una combinación lineal sencilla:

$$
\text{RADAR}(j, \ell) = \alpha \cdot C^*(j, \ell) + \beta \cdot \text{IPC}(j) + \gamma \cdot \text{Tendencia}(j)
$$

donde $j$ es el país, $\ell$ es la línea de negocio (para el score por línea) o nulo (para el score global), y los pesos $\alpha + \beta + \gamma = 1$. Los valores por defecto del sistema son $\alpha = 0.55$, $\beta = 0.30$ y $\gamma = 0.15$, derivados de la mediana de la literatura sistematizada en la revisión bibliográfica del proyecto.

## La interpretación de los tres pesos

Cada uno de los tres pesos cuenta una historia estratégica diferente sobre cómo Cibest mira los mercados. **$\alpha$ pondera el atractivo absoluto del país** medido con TOPSIS. Si Cibest fuera puramente "globalista", $\alpha$ tendería a 1 y los otros dos tenderían a 0: solo importaría qué tan grande, desarrollado e institucionalmente sólido es cada mercado. **$\beta$ pondera la afinidad bilateral con Colombia**. Si Cibest fuera puramente "regionalista", $\beta$ tendería a 1: solo importaría qué tan cercano está el mercado en sentido amplio. **$\gamma$ pondera el dinamismo reciente**. Su valor por defecto es bajo (15%) porque la decisión de internacionalización es de largo plazo y no debería estar dominada por el ciclo, pero suficiente para que un mercado en plena recuperación pueda destacarse frente a otro estancado con scores similares.

Los valores por defecto $(0.55, 0.30, 0.15)$ representan una postura intermedia: el atractivo absoluto pesa más de la mitad, la proximidad es relevante pero no decisiva, y el dinamismo modula. Estos valores se pueden ajustar en el dashboard mediante sliders, lo que permite a la dirección explorar escenarios "qué pasa si fuéramos más globalistas" o "qué pasa si priorizáramos más a los vecinos".

## Cálculo del score RADAR sobre nuestro ejemplo

In [ ]:
# Score RADAR para los cinco paises
# Componente 1: C* viene de TOPSIS (calculado en capitulo 5)
C_star = cc.copy()
# Componente 2: IPC (calculado en capitulo 6)
IPC = ipc.copy()
# Componente 3: tendencia. Para el ejemplo usamos crecimiento del PIB sintetico
tendencia_raw = pd.Series({
    'COL': 2.5, 'MEX': 3.0, 'CHL': 1.8, 'PAN': 7.5, 'ESP': 2.2,
})
# Normalizar tendencia a [0,1]
tendencia = (tendencia_raw - tendencia_raw.min()) / (tendencia_raw.max() - tendencia_raw.min())

# Pesos por defecto del sistema
alpha, beta, gamma = 0.55, 0.30, 0.15

score_radar = alpha * C_star + beta * IPC + gamma * tendencia

resumen = pd.DataFrame({
    'C* (TOPSIS)': C_star.round(3),
    'IPC (Prox.)': IPC.round(3),
    'Tendencia':   tendencia.round(3),
    'RADAR':       score_radar.round(3),
})
resumen = resumen.sort_values('RADAR', ascending=False)
resumen.insert(0, 'Rank', range(1, len(resumen) + 1))

print(f'Score RADAR compuesto (α={alpha}, β={beta}, γ={gamma}):')
style_table(resumen,
            gradient_cols=['C* (TOPSIS)', 'IPC (Prox.)', 'Tendencia', 'RADAR'],
            gradient_cmap=cmap_custom,
            format_dict={'C* (TOPSIS)': '{:.3f}', 'IPC (Prox.)': '{:.3f}',
                         'Tendencia': '{:.3f}', 'RADAR': '{:.3f}', 'Rank': '{:.0f}'})


Score RADAR compuesto (α=0.55, β=0.3, γ=0.15):


,Rank,C* (TOPSIS),IPC (Prox.),Tendencia,RADAR
PAN,1,0.485,0.700,1.000,0.626
COL,2,0.343,1.000,0.123,0.507
CHL,3,0.575,0.485,0.000,0.462
ESP,4,0.654,0.250,0.070,0.445
MEX,5,0.294,0.533,0.211,0.353


In [ ]:
# Visualizacion stacked del score RADAR mostrando contribucion de cada componente
fig = go.Figure()
df_stack = resumen.sort_values('RADAR')

fig.add_trace(go.Bar(
    name='α · C* (TOPSIS)',
    y=df_stack.index, x=alpha * df_stack['C* (TOPSIS)'],
    orientation='h', marker_color=CIBEST['gray'],
    hovertemplate='<b>%{y}</b><br>α·C* = %{x:.3f}<extra></extra>',
))
fig.add_trace(go.Bar(
    name='β · IPC (Proximidad)',
    y=df_stack.index, x=beta * df_stack['IPC (Prox.)'],
    orientation='h', marker_color=CIBEST['yellow'],
    hovertemplate='<b>%{y}</b><br>β·IPC = %{x:.3f}<extra></extra>',
))
fig.add_trace(go.Bar(
    name='γ · Tendencia',
    y=df_stack.index, x=gamma * df_stack['Tendencia'],
    orientation='h', marker_color=CIBEST['gray_light'],
    hovertemplate='<b>%{y}</b><br>γ·Tendencia = %{x:.3f}<extra></extra>',
))

fig.update_layout(
    title=f'<b>Descomposición del score RADAR</b><br>'
          f'<sub>RADAR = α·C* + β·IPC + γ·Tendencia · '
          f'α={alpha}, β={beta}, γ={gamma}</sub>',
    barmode='stack',
    xaxis=dict(title='Contribución al score RADAR', range=[0, 1], gridcolor=CIBEST['gray_border']),
    yaxis_title='País',
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
    height=400, legend=dict(orientation='h', y=-0.2),
    margin=dict(l=100),
)
fig.show()


La gráfica anterior es muy informativa. La parte oscura es lo que aporta el atractivo absoluto del país (C*), la parte amarilla es lo que aporta la proximidad con Colombia (IPC), y la gris es lo que aporta el dinamismo. Mirar el patrón de las barras ayuda a entender por qué cada país está donde está: España rankea alto principalmente por su C* (atractivo absoluto), Panamá rankea alto principalmente por su IPC (proximidad), Colombia rankea muy alto en proximidad por construcción pero su C* es modesto. Esa decomposición visual es exactamente lo que el dashboard ofrece a la alta dirección para que pueda explicar el ranking y argumentar decisiones.

---

# Capítulo 8 — Motor de señales por línea de negocio

## De scores numéricos a etiquetas accionables

Hasta este punto el sistema produce números: scores RADAR entre 0 y 1, rankings de 1 a 30. Esos números son útiles para los analistas, pero para que el resultado sea verdaderamente *accionable* por la alta dirección, conviene convertirlos en **etiquetas categóricas claras**: ¿este país es una alta oportunidad, una oportunidad moderada, una baja oportunidad, o un riesgo? La transformación de scores en etiquetas la hace el **motor de señales**, que aplica un sistema simple de cuatro niveles basado en percentiles, con una capa adicional de "overrides de riesgo" para garantizar que ningún mercado con problemas institucionales severos se etiquete como oportunidad alta solo por tener un score numérico bueno.

## Las cuatro etiquetas y sus umbrales por defecto

Los umbrales del sistema están definidos sobre el percentil del score por línea, no sobre el valor absoluto del score. Esa decisión es importante: si los umbrales fueran absolutos (digamos "ALTA si C* > 0.7"), un cambio uniforme en la matriz de decisión podría hacer que ningún país calificara como ALTA en una corrida y todos en la siguiente. Usar percentiles asegura que las señales sean *relativas al conjunto evaluado* y por tanto más estables operativamente.

In [ ]:
umbrales_senales = pd.DataFrame([
    {'Señal': 'ALTA_OPORTUNIDAD',     'Color': '🟢',
     'Condición': 'Percentil del score ≥ 0.70 y sin overrides activos',
     'Interpretación estratégica': 'Mercado prioritario para esta línea — abrir conversaciones'},
    {'Señal': 'OPORTUNIDAD_MODERADA', 'Color': '🟡',
     'Condición': '0.45 ≤ Percentil < 0.70 y sin overrides',
     'Interpretación estratégica': 'Mercado de segundo orden — monitorear y profundizar análisis'},
    {'Señal': 'BAJA_OPORTUNIDAD',     'Color': '🟠',
     'Condición': '0.25 ≤ Percentil < 0.45',
     'Interpretación estratégica': 'Mercado no prioritario en el ciclo actual'},
    {'Señal': 'RIESGO',               'Color': '🔴',
     'Condición': 'Percentil < 0.25 ó cualquier override activo',
     'Interpretación estratégica': 'Mercado a evitar o descartar — riesgo institucional o de score'},
])
style_table(umbrales_senales)


,Señal,Color,Condición,Interpretación estratégica
0,ALTA_OPORTUNIDAD,🟢,Percentil del score ≥ 0.70 y sin overrides activos,Mercado prioritario para esta línea — abrir conversaciones
1,OPORTUNIDAD_MODERADA,🟡,0.45 ≤ Percentil < 0.70 y sin overrides,Mercado de segundo orden — monitorear y profundizar análisis
2,BAJA_OPORTUNIDAD,🟠,0.25 ≤ Percentil < 0.45,Mercado no prioritario en el ciclo actual
3,RIESGO,🔴,Percentil < 0.25 ó cualquier override activo,Mercado a evitar o descartar — riesgo institucional o de score


## Los overrides de riesgo

Existen dos *overrides* automáticos que fuerzan la señal a RIESGO sin importar el score numérico. Si la **estabilidad política** (indicador WGI `PV.EST`) está en el percentil inferior al 15%, el país se etiqueta como RIESGO. Lo mismo si el **control de la corrupción** (indicador WGI `CC.EST`) está en el percentil inferior al 15%. Estos overrides son una salvaguarda: aseguran que un país con problemas institucionales severos no pueda terminar como "oportunidad alta" porque sus otras variables sean buenas.

## Cinco rankings, cinco señales: el resultado diferenciado por línea

Como recordatorio: TOPSIS se ejecuta una vez por cada línea con su `weight_profile` correspondiente. Eso produce cinco scores por país. Y el motor de señales se aplica cinco veces sobre cada uno de esos scores. El resultado final es una **matriz país × línea con etiquetas categóricas**, que es justamente lo que el Comité necesita para tomar decisiones: en lugar de ver "Panamá tiene C* de 0.62 para PT y 0.48 para BD", ve "Panamá es ALTA para PT y MODERADA para BD".

In [ ]:
# Ejemplo de matriz de senales con los cinco paises del notebook (sinteticos)
# Para no inventar pesos por linea complejos en este ejemplo, asumimos scores hipoteticos
np.random.seed(42)
scores_por_linea = pd.DataFrame({
    'IB':  [0.55, 0.45, 0.78, 0.62, 0.85],   # Colombia, Mexico, Chile, Panama, Espana
    'PF':  [0.50, 0.70, 0.60, 0.82, 0.75],
    'AD':  [0.45, 0.55, 0.40, 0.50, 0.70],
    'BD':  [0.52, 0.68, 0.65, 0.48, 0.80],
    'CIB': [0.48, 0.72, 0.75, 0.58, 0.88],
}, index=['COL', 'MEX', 'CHL', 'PAN', 'ESP'])

def clasificar_senal(percentil):
    if percentil >= 0.70: return 'ALTA_OPORTUNIDAD'
    if percentil >= 0.45: return 'OPORTUNIDAD_MODERADA'
    if percentil >= 0.25: return 'BAJA_OPORTUNIDAD'
    return 'RIESGO'

senales = pd.DataFrame(index=scores_por_linea.index, columns=scores_por_linea.columns)
for linea in scores_por_linea.columns:
    percentiles = scores_por_linea[linea].rank(pct=True)
    senales[linea] = percentiles.apply(clasificar_senal)

print('📡 Matriz de señales país × línea de negocio:')
display(senales)


📡 Matriz de señales país × línea de negocio:


,IB,PF,AD,BD,CIB
COL,BAJA_OPORTUNIDAD,RIESGO,BAJA_OPORTUNIDAD,BAJA_OPORTUNIDAD,RIESGO
MEX,RIESGO,OPORTUNIDAD_MODERADA,ALTA_OPORTUNIDAD,ALTA_OPORTUNIDAD,OPORTUNIDAD_MODERADA
CHL,ALTA_OPORTUNIDAD,BAJA_OPORTUNIDAD,RIESGO,OPORTUNIDAD_MODERADA,ALTA_OPORTUNIDAD
PAN,OPORTUNIDAD_MODERADA,ALTA_OPORTUNIDAD,OPORTUNIDAD_MODERADA,RIESGO,BAJA_OPORTUNIDAD
ESP,ALTA_OPORTUNIDAD,ALTA_OPORTUNIDAD,ALTA_OPORTUNIDAD,ALTA_OPORTUNIDAD,ALTA_OPORTUNIDAD


In [ ]:
# Heatmap categorico de las senales
emoji_map = {'ALTA_OPORTUNIDAD': '🟢', 'OPORTUNIDAD_MODERADA': '🟡',
             'BAJA_OPORTUNIDAD': '🟠', 'RIESGO': '🔴'}
codigo_map = {'ALTA_OPORTUNIDAD': 4, 'OPORTUNIDAD_MODERADA': 3,
              'BAJA_OPORTUNIDAD': 2, 'RIESGO': 1}
senales_num = senales.replace(codigo_map).astype(int)
senales_text = senales.apply(lambda c: c.map(lambda v: emoji_map[v]))

fig = go.Figure(data=go.Heatmap(
    z=senales_num.values,
    x=senales_num.columns, y=senales_num.index,
    text=senales_text.values, texttemplate='%{text}',
    textfont=dict(size=24),
    colorscale=[[0, CIBEST['red']], [0.33, CIBEST['amber']],
                [0.66, CIBEST['gold']], [1, CIBEST['green']]],
    showscale=False,
    hovertemplate='<b>%{y}</b> · <b>%{x}</b><br>Señal: %{text}<extra></extra>',
))
fig.update_layout(
    title='<b>Heatmap de señales · país × línea de negocio</b><br>'
          '<sub>🟢 Alta · 🟡 Moderada · 🟠 Baja · 🔴 Riesgo</sub>',
    xaxis=dict(title='Línea de negocio', side='top'),
    yaxis=dict(title='País', autorange='reversed'),
    height=400, width=700,
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    font=dict(family='Arial', color=CIBEST['gray']),
)
fig.show()


Esta es la salida más usada del dashboard ejecutivo. Permite ver, de un solo golpe de vista, qué países son prioritarios para qué líneas. Las narrativas automáticas que se generan en la fase B del proyecto (mayo-junio 2026) toman esta matriz como insumo principal y la enriquecen con contexto cualitativo de fuentes externas.

---

# Capítulo 9 — Validación: análisis de sensibilidad y TOPSIS vs VIKOR

## ¿Por qué validar?

Un ranking que cambia drásticamente cuando los pesos cambian un poco es un ranking frágil, y un ranking frágil no es base para una decisión estratégica de largo plazo. La validación de RADAR Cibest tiene dos componentes complementarios: **análisis de sensibilidad** y **validación cruzada con un método alternativo**.

## Análisis de sensibilidad: perturbar pesos y ver qué cambia

La idea del análisis de sensibilidad es muy simple: tomamos los pesos óptimos producidos por BWM y los perturbamos sistemáticamente. Por ejemplo, multiplicamos el peso de la dimensión Macro por 0.80 (lo reducimos 20%) y renormalizamos para que la suma siga siendo 1; recalculamos TOPSIS con esos pesos perturbados; comparamos el ranking resultante con el original. Si los rankings son muy similares (correlación de Spearman ≥ 0.85) y los top-N comparten al menos 70-80% de países, el ranking es *robusto* ante esa perturbación. Si los rankings se mezclan drásticamente, el ranking es *frágil* en esa dirección y conviene reportar el hallazgo a la dirección.

El sistema repite este ejercicio para cada dimensión y para varios niveles de perturbación (±10%, ±20%), produciendo una tabla resumen que muestra qué tan robusto es el ranking en cada dirección.

In [ ]:
# Simulacion del analisis de sensibilidad sobre el ejemplo
def topsis_score(matriz_norm, pesos):
    """TOPSIS abreviado para la simulacion."""
    ponderada = matriz_norm.copy()
    for c in ponderada.columns:
        ponderada[c] = ponderada[c] * pesos[c]
    A_pos = ponderada.max()
    A_neg = ponderada.min()
    d_p = np.sqrt(((ponderada - A_pos) ** 2).sum(axis=1))
    d_n = np.sqrt(((ponderada - A_neg) ** 2).sum(axis=1))
    return d_n / (d_p + d_n)

# Pesos base (uniformes para el ejemplo)
pesos_base = {c: 1/len(matriz_norm.columns) for c in matriz_norm.columns}
score_base = topsis_score(matriz_norm, pesos_base)

# Perturbar cada variable +/- 20%
resultados_sens = []
for var in matriz_norm.columns:
    for pert in [0.80, 0.90, 1.10, 1.20]:
        pesos_p = pesos_base.copy()
        pesos_p[var] = pesos_p[var] * pert
        s = sum(pesos_p.values())
        pesos_p = {k: v/s for k, v in pesos_p.items()}
        score_p = topsis_score(matriz_norm, pesos_p)
        corr = score_base.corr(score_p, method='spearman')
        resultados_sens.append({
            'Variable perturbada': var,
            'Perturbación':        f'×{pert:.2f}',
            'Corr. Spearman':      round(corr, 4),
        })

df_sens = pd.DataFrame(resultados_sens)
print('🔬 Resultados del análisis de sensibilidad (5 variables × 4 perturbaciones):')
style_table(df_sens.head(15), gradient_cols=['Corr. Spearman'], gradient_cmap=cmap_custom,
            format_dict={'Corr. Spearman': '{:.4f}'})


🔬 Resultados del análisis de sensibilidad (5 variables × 4 perturbaciones):


,Variable perturbada,Perturbación,Corr. Spearman
0,gdp_per_capita_ppp,×0.80,1.0000
1,gdp_per_capita_ppp,×0.90,1.0000
2,gdp_per_capita_ppp,×1.10,1.0000
3,gdp_per_capita_ppp,×1.20,1.0000
4,inflation_rate,×0.80,1.0000
5,inflation_rate,×0.90,1.0000
6,inflation_rate,×1.10,1.0000
7,inflation_rate,×1.20,1.0000
8,financial_system_deposits_gdp,×0.80,1.0000
9,financial_system_deposits_gdp,×0.90,1.0000


Una correlación de Spearman cercana a 1 entre el ranking original y el ranking perturbado significa que las posiciones de los países cambian poco bajo la perturbación: el ranking es robusto en esa dirección. Una correlación que baja de 0.85 sugiere que la dimensión perturbada tiene poder discriminante alto y vale la pena reportar al Comité ese hecho explícitamente para que sepan dónde está la "fragilidad" del modelo.

## Validación cruzada: TOPSIS contra VIKOR

La segunda capa de validación es metodológica: ¿qué pasaría si en lugar de TOPSIS hubiéramos usado un método multicriterio diferente? Si el ranking resultante fuera muy distinto, eso sugeriría que el ranking depende fuertemente de la elección del método y no del fenómeno subyacente, lo cual debilitaría las conclusiones. El sistema usa **VIKOR** (Opricovic y Tzeng, 2004) como método alternativo para esta validación cruzada porque comparte filosofía con TOPSIS (busca el "compromiso" más cercano al ideal) pero usa una métrica de distancia diferente.

VIKOR define dos medidas para cada país: $S_i$ (la "utilidad agregada" que suma las distancias ponderadas al ideal en todas las dimensiones) y $R_i$ (la "máxima distancia" individual al ideal en cualquier dimensión). El score de compromiso $Q_i$ combina ambas:

$$
Q_i = v \cdot \frac{S_i - S^*}{S^- - S^*} + (1-v) \cdot \frac{R_i - R^*}{R^- - R^*}
$$

donde $v$ es un parámetro de estrategia (típicamente 0.5) y $S^*, S^-, R^*, R^-$ son los mejores y peores valores observados. La correlación de Spearman entre el ranking TOPSIS y el ranking VIKOR debe ser **al menos 0.85** para que el resultado se considere robusto metodológicamente. Si fuera más baja, habría que reportar al Comité que la elección del método multicriterio está influyendo en las conclusiones y conviene profundizar.

## Lo que la validación logra y lo que no

Es importante ser honesto sobre los límites de la validación. Lo que la validación **sí** logra es identificar si el ranking es robusto ante cambios menores en los pesos y ante cambios de método. Eso da seguridad de que las conclusiones no son artefactos de decisiones técnicas particulares. Lo que la validación **no** logra es validar que las *variables* elegidas son las correctas o que la *teoría* subyacente es adecuada para el problema de negocio. Esa parte se valida en las fases 1 y 2 de ASUM-DM (entendimiento del negocio y entendimiento de los datos), no en la fase 5 de evaluación. Es por eso que el rigor metodológico de las primeras fases es tan importante como el rigor analítico de las posteriores.

---

# Capítulo 10 — Recapitulación visual del flujo completo

## El recorrido entero, en una sola figura

Hemos llegado al final del notebook. Para cerrar, vale la pena recapitular el flujo completo del sistema en una sola visualización que sintetiza todo lo que hemos visto. La figura siguiente muestra cómo cada técnica que estudiamos en los capítulos anteriores se conecta con las demás y produce, al final, la información accionable que llega al Comité Ejecutivo y a la Junta Directiva.

In [ ]:
# Diagrama de capas con la metodologia completa
fig = make_subplots(rows=1, cols=1)

capas = [
    ('1. Project Charter + revisión de literatura',                    0,  CIBEST['gray_bg'],    CIBEST['gray']),
    ('2. Diccionario de variables + extracción de datos',              1,  CIBEST['gray_light'], 'white'),
    ('3. Limpieza · imputación · normalización · dirección',          2,  CIBEST['gray_light'], 'white'),
    ('4a. BWM con panel ejecutivo → pesos de dimensiones y variables', 3,  CIBEST['gray'],       CIBEST['yellow']),
    ('4b. TOPSIS global + cinco TOPSIS por línea de negocio',         4,  CIBEST['gray'],       CIBEST['yellow']),
    ('4c. Modelo gravitacional → IPC',                                 5,  CIBEST['gray'],       CIBEST['yellow']),
    ('4d. Score compuesto: RADAR = α·C* + β·IPC + γ·Tendencia',       6,  CIBEST['gold'],       CIBEST['gray']),
    ('4e. Motor de señales (4 niveles × 5 líneas)',                    7,  CIBEST['gold_dark'],  'white'),
    ('5. Sensibilidad + validación cruzada TOPSIS↔VIKOR',             8,  CIBEST['green'],      'white'),
    ('Entrega: reporte estratégico + dashboard + narrativas IA',       9,  CIBEST['red'],        'white'),
]

for nombre, y, color, text_color in capas:
    fig.add_shape(
        type='rect', x0=0, x1=10, y0=y - 0.4, y1=y + 0.4,
        line=dict(color=CIBEST['gray'], width=1.5),
        fillcolor=color,
    )
    fig.add_annotation(
        x=5, y=y, text=f'<b>{nombre}</b>', showarrow=False,
        font=dict(family='Arial', size=13, color=text_color),
    )

# Flechas entre capas
for y in range(len(capas) - 1):
    fig.add_annotation(
        x=5, y=y + 0.55, ax=5, ay=y + 0.4,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=2,
        arrowcolor=CIBEST['gray'],
    )

fig.update_layout(
    title='<b>RADAR Cibest · arquitectura metodológica completa</b><br>'
          '<sub>De Project Charter (abajo) a entrega ejecutiva (arriba)</sub>',
    xaxis=dict(visible=False, range=[-0.5, 10.5]),
    yaxis=dict(visible=False, range=[-1, len(capas)]),
    plot_bgcolor=CIBEST['white'], paper_bgcolor=CIBEST['white'],
    height=720, width=950, showlegend=False,
    margin=dict(t=80, b=30, l=30, r=30),
)
fig.show()


## Las ideas que vale la pena llevarse del notebook

La primera idea es que **RADAR Cibest no es una sola técnica sino una integración cuidadosa de varias técnicas**, cada una eligida para resolver un sub-problema específico. BWM resuelve el problema de capturar el juicio experto de forma estructurada con poca carga cognitiva. TOPSIS resuelve el problema de combinar muchas variables heterogéneas en un ranking interpretable. El modelo gravitacional resuelve el problema de incorporar la dimensión bilateral con Colombia. El motor de señales resuelve el problema de hacer accionable un resultado numérico para audiencias ejecutivas. Sustituir cualquiera de esas piezas por otra cambiaría la naturaleza del sistema.

**¿Por qué un modelo multicriterio hibrido? ¿No es demasiado complejo? ¿Por qué no usar simplemente un enfoque de proximidad (países cercanos a Colombia)?**

El negocio ya es complejo. El error es simplificarlo mal.

Un holding financiero:
opera bajo regulación,
gestiona riesgo,
depende de estructura bancaria,
y, en tres líneas, de adopción digital.

La literatura es clara:

Los modelos de selección de mercados de una sola dimensión fallan sistemáticamente en servicios financieros.
la proximidad reduce fricción, pero no maximiza rentabilidad.

**Nuestro problema no es dónde es más fácil entrar**, sino dónde tenemos mayor probabilidad de generar retornos sostenibles por línea de negocio.

La evidencia muestra que:

- La distancia sí importa, pero explica fricción, *no atractivo económico total*. La distancia reduce flujos, pero no los explica completamente.
- El tamaño del mercado, la calidad institucional y la profundidad financiera explican más varianza en flujos financieros que la cercanía por sí sola. Bancos exitosos como BBVA o Santander combinaron proximidad cultural con mercados grandes y financieramente profundos.

RADAR no elimina la proximidad: la integra como un componente explícito, sin permitir que opaque mercados grandes, rentables o estratégicos.

La segunda idea es que **las cinco líneas de negocio se atienden simultáneamente** gracias a la ejecución múltiple de TOPSIS con perfiles de pesos diferenciados. Esa decisión metodológica responde directamente al Gap 2 identificado en la revisión sistemática de literatura del proyecto, que documentó que ningún sistema publicado de evaluación de mercados produce señales diferenciadas por tipo de negocio financiero. Ese diferencial es propio de RADAR Cibest.

La tercera idea es que **la robustez del ranking se valida formalmente** mediante análisis de sensibilidad y validación cruzada con VIKOR. Esa validación no es un trámite: es lo que da soporte a la dirección para presentar el resultado ante la Junta sabiendo que no se desmoronará ante una pregunta del tipo "¿y si nos equivocamos un poco con los pesos?".

La cuarta idea es que **el sistema es modular y configurable**. Todas las decisiones que se pueden razonablemente discutir (qué variables incluir, qué pesos asignar, cuáles son los umbrales de señal, qué valores tienen $\alpha$, $\beta$ y $\gamma$) están expuestas en archivos YAML editables. Cambiar la estrategia de Cibest hacia una postura más globalista o más regionalista no requiere reescribir código: requiere ajustar un par de números en `config/settings.yaml` y volver a correr el pipeline. Esa modularidad es lo que va a permitir que el sistema sobreviva al ciclo estratégico anual de revisión con la Junta Directiva durante años.

---

## Próximos pasos en el proyecto

Para cerrar, vale la pena recordar dónde estamos en el cronograma del proyecto. La fase A (MVP cuantitativo, marzo-mayo 2026) está prácticamente concluida con la entrega del sistema completo. La fase B (narrativas ejecutivas enriquecidas con LLM, mayo-junio 2026) se inicia en las próximas semanas. La sesión de elicitación BWM con la alta dirección está sujeta a disponibilidad, usando exclusivamente herramientas Microsoft Office.

Gracias por leer este notebook hasta el final.

---

*Notebook 00 — versión 1.0 · Dirección de Estrategia · Grupo Cibest · 2026*